# Notebook 6: Streams + Tasks for Sub-15s Feature Freshness

This notebook explores an alternative to Dynamic Tables for feature engineering: using **Streams and Tasks** to achieve the fastest possible feature refresh (10-15 seconds).

---

### Why This Notebook Exists

In Notebook 2, we built features with Dynamic Tables (TARGET_LAG = 1 minute). The measured freshness was 33-39 seconds. For most fraud use cases, this is sufficient.

However, if the customer requires **sub-15 second freshness** (e.g., blocking the 2nd transaction in a rapid card-testing attack), Streams + Tasks offer a faster path at the cost of increased complexity.

---

### Dynamic Tables vs Streams + Tasks

| | Dynamic Tables (NB02) | Streams + Tasks (this notebook) |
|---|---|---|
| Feature freshness | 33-60 seconds | 10-15 seconds |
| Implementation | Declarative (write a SELECT) | Imperative (write MERGE logic) |
| Maintenance | Zero (Snowflake manages) | Medium (you own the logic) |
| Rolling window handling | Automatic (full recompute) | Manual (add + expire logic) |
| Error recovery | Automatic (retry on failure) | Manual (Task error handling) |
| Cost | $13,190/month (Medium WH 24/7) | Similar (warehouse still always-on) |
| Recommended for | Most use cases | Sub-15s SLA requirement only |

---

### Architecture

```
FRAUD_TRANSACTIONS table
        |
        v
    [Stream] (captures new rows instantly, zero lag)
        |
        v
    [Task - runs every 10s (Snowflake minimum)]
        |
        v
    MERGE INTO feature tables (incremental update, all 5 entities)
        |
        v
    Features available for scoring within 10-15s of INSERT
```

---

### Key Challenge: Rolling Window Expiry

The hardest part of Streams + Tasks for time-windowed features is **expiry**. Consider `PURCHASES_NUM_L1H` (purchases in last 1 hour):

- When a new transaction arrives: INCREMENT the count (easy)
- When a transaction ages past 1 hour: DECREMENT the count (hard)

With Dynamic Tables, this is automatic (full recompute every cycle). With Tasks, you need a separate expiry mechanism.

**Our approach:** Use a dual-task pattern:
1. **Fast Task (10s):** Incremental MERGE for new transactions (all 5 entities)
2. **Slow Task (5 min):** Full recompute of window aggregates (handles expiry, all 5 entities)

This gives 10-15s freshness for NEW activity while keeping windows accurate within 5 minutes.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
session.sql("USE ROLE ACCOUNTADMIN").collect()
session.sql("USE WAREHOUSE FRAUD_DEMO_WH").collect()
session.sql("USE DATABASE FRAUD_DEMO_DEV").collect()
session.sql("USE SCHEMA FEATURES").collect()
print("Context: FRAUD_DEMO_DEV.FEATURES (ACCOUNTADMIN)")

## 6.1 Create the Stream

A Stream tracks changes (inserts, updates, deletes) on the source table. It has **zero latency** - changes are visible the instant they are committed. The Stream is the "trigger" that tells the Task new data is available.

In [ ]:
%%sql -r stream_result
-- Create a Stream on the transactions table.
-- APPEND_ONLY = TRUE because we only INSERT transactions (never update/delete them).
-- This is more efficient than tracking all DML types.
CREATE OR REPLACE STREAM FRAUD_DEMO_DEV.FEATURES.TXN_STREAM
    ON TABLE FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS
    APPEND_ONLY = TRUE
    COMMENT = 'Captures new transactions for real-time feature updates';

## 6.2 Create the Hybrid Feature Tables (All 5 Entities)

We create **Hybrid Tables** directly — not standard tables. This means:
- The fast task MERGEs into them (sub-10ms writes per row)
- The scoring service reads from them (sub-10ms point lookups by primary key)
- No intermediate copy step needed

Hybrid Tables enforce PRIMARY KEYs and use row-based indexes, giving us both fast writes (from the Task) and fast reads (from the scoring endpoint) in a single table.

| Standard Table | Hybrid Table (what we use) |
|---|---|
| Columnar storage | Row-based indexed storage |
| Fast for analytics/scans | Fast for single-row CRUD |
| Point lookup: 200-500ms (warehouse query) | Point lookup: 5-10ms (index-based) |
| No enforced PRIMARY KEY | Enforced PRIMARY KEY |

We create tables for all 5 entities matching the Dynamic Tables in NB02:
1. **CUSTOMER_VELOCITY_RT** - 57 features (5 windows x 11 metrics)
2. **MERCHANT_VELOCITY_RT** - 20 features
3. **WALLET_DPAN_VELOCITY_RT** - 20 features
4. **IP_VELOCITY_RT** - 20 features
5. **CUSTOMER_MERCHANT_VELOCITY_RT** - 10 features

In [ ]:
-- All 5 entity feature tables as HYBRID TABLES.
-- PRIMARY KEY enables sub-10ms point lookups at scoring time.
-- The fast task MERGEs directly into these (no intermediate tables needed).

-- Entity 1: Customer Velocity (Hybrid)
CREATE OR REPLACE HYBRID TABLE FRAUD_DEMO_DEV.FEATURES.CUSTOMER_VELOCITY_RT (
    customer_id VARCHAR PRIMARY KEY,
    last_updated_ts TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    latest_txn_ts TIMESTAMP_NTZ,
    purchases_num_l1h NUMBER DEFAULT 0, purchases_sum_l1h FLOAT DEFAULT 0, purchases_min_l1h FLOAT, purchases_max_l1h FLOAT DEFAULT 0,
    distinct_purchase_amt_l1h NUMBER DEFAULT 0, purchases_gbr_num_l1h NUMBER DEFAULT 0, purchases_nongbr_num_l1h NUMBER DEFAULT 0,
    distinct_card_tokens_l1h NUMBER DEFAULT 0, distinct_wallet_dpan_l1h NUMBER DEFAULT 0, distinct_merchants_l1h NUMBER DEFAULT 0, declined_num_l1h NUMBER DEFAULT 0,
    purchases_num_l6h NUMBER DEFAULT 0, purchases_sum_l6h FLOAT DEFAULT 0, purchases_min_l6h FLOAT, purchases_max_l6h FLOAT DEFAULT 0,
    distinct_purchase_amt_l6h NUMBER DEFAULT 0, purchases_gbr_num_l6h NUMBER DEFAULT 0, purchases_nongbr_num_l6h NUMBER DEFAULT 0,
    distinct_card_tokens_l6h NUMBER DEFAULT 0, distinct_wallet_dpan_l6h NUMBER DEFAULT 0, distinct_merchants_l6h NUMBER DEFAULT 0, declined_num_l6h NUMBER DEFAULT 0,
    purchases_num_l24h NUMBER DEFAULT 0, purchases_sum_l24h FLOAT DEFAULT 0, purchases_min_l24h FLOAT, purchases_max_l24h FLOAT DEFAULT 0,
    distinct_purchase_amt_l24h NUMBER DEFAULT 0, purchases_gbr_num_l24h NUMBER DEFAULT 0, purchases_nongbr_num_l24h NUMBER DEFAULT 0,
    distinct_card_tokens_l24h NUMBER DEFAULT 0, distinct_wallet_dpan_l24h NUMBER DEFAULT 0, distinct_merchants_l24h NUMBER DEFAULT 0, distinct_countries_l24h NUMBER DEFAULT 0, declined_num_l24h NUMBER DEFAULT 0,
    purchases_num_l48h NUMBER DEFAULT 0, purchases_sum_l48h FLOAT DEFAULT 0, purchases_min_l48h FLOAT, purchases_max_l48h FLOAT DEFAULT 0,
    distinct_purchase_amt_l48h NUMBER DEFAULT 0, purchases_gbr_num_l48h NUMBER DEFAULT 0, purchases_nongbr_num_l48h NUMBER DEFAULT 0,
    distinct_card_tokens_l48h NUMBER DEFAULT 0, distinct_wallet_dpan_l48h NUMBER DEFAULT 0, distinct_merchants_l48h NUMBER DEFAULT 0, declined_num_l48h NUMBER DEFAULT 0,
    purchases_num_l1wk NUMBER DEFAULT 0, purchases_sum_l1wk FLOAT DEFAULT 0, purchases_min_l1wk FLOAT, purchases_max_l1wk FLOAT DEFAULT 0,
    distinct_purchase_amt_l1wk NUMBER DEFAULT 0, purchases_gbr_num_l1wk NUMBER DEFAULT 0, purchases_nongbr_num_l1wk NUMBER DEFAULT 0,
    distinct_card_tokens_l1wk NUMBER DEFAULT 0, distinct_wallet_dpan_l1wk NUMBER DEFAULT 0, distinct_merchants_l1wk NUMBER DEFAULT 0, distinct_countries_l1wk NUMBER DEFAULT 0, declined_num_l1wk NUMBER DEFAULT 0
);

-- Entity 2: Merchant Velocity (Hybrid)
CREATE OR REPLACE HYBRID TABLE FRAUD_DEMO_DEV.FEATURES.MERCHANT_VELOCITY_RT (
    merchant_id VARCHAR PRIMARY KEY,
    last_updated_ts TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    latest_txn_ts TIMESTAMP_NTZ,
    merchant_purchases_num_l1h NUMBER DEFAULT 0, merchant_purchases_sum_l1h FLOAT DEFAULT 0,
    merchant_distinct_customers_l1h NUMBER DEFAULT 0, merchant_declined_num_l1h NUMBER DEFAULT 0,
    merchant_purchases_num_l6h NUMBER DEFAULT 0, merchant_purchases_sum_l6h FLOAT DEFAULT 0,
    merchant_distinct_customers_l6h NUMBER DEFAULT 0, merchant_declined_num_l6h NUMBER DEFAULT 0,
    merchant_purchases_num_l24h NUMBER DEFAULT 0, merchant_purchases_sum_l24h FLOAT DEFAULT 0,
    merchant_distinct_customers_l24h NUMBER DEFAULT 0, merchant_declined_num_l24h NUMBER DEFAULT 0,
    merchant_purchases_num_l48h NUMBER DEFAULT 0, merchant_purchases_sum_l48h FLOAT DEFAULT 0,
    merchant_distinct_customers_l48h NUMBER DEFAULT 0, merchant_declined_num_l48h NUMBER DEFAULT 0,
    merchant_purchases_num_l1wk NUMBER DEFAULT 0, merchant_purchases_sum_l1wk FLOAT DEFAULT 0,
    merchant_distinct_customers_l1wk NUMBER DEFAULT 0, merchant_declined_num_l1wk NUMBER DEFAULT 0
);

-- Entity 3: DPAN Velocity (Hybrid)
CREATE OR REPLACE HYBRID TABLE FRAUD_DEMO_DEV.FEATURES.WALLET_DPAN_VELOCITY_RT (
    wallet_dpan VARCHAR PRIMARY KEY,
    last_updated_ts TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    latest_txn_ts TIMESTAMP_NTZ,
    dpan_purchases_num_l1h NUMBER DEFAULT 0, dpan_purchases_sum_l1h FLOAT DEFAULT 0,
    dpan_distinct_merchants_l1h NUMBER DEFAULT 0, dpan_declined_num_l1h NUMBER DEFAULT 0,
    dpan_purchases_num_l6h NUMBER DEFAULT 0, dpan_purchases_sum_l6h FLOAT DEFAULT 0,
    dpan_distinct_merchants_l6h NUMBER DEFAULT 0, dpan_declined_num_l6h NUMBER DEFAULT 0,
    dpan_purchases_num_l24h NUMBER DEFAULT 0, dpan_purchases_sum_l24h FLOAT DEFAULT 0,
    dpan_distinct_merchants_l24h NUMBER DEFAULT 0, dpan_declined_num_l24h NUMBER DEFAULT 0,
    dpan_purchases_num_l48h NUMBER DEFAULT 0, dpan_purchases_sum_l48h FLOAT DEFAULT 0,
    dpan_distinct_merchants_l48h NUMBER DEFAULT 0, dpan_declined_num_l48h NUMBER DEFAULT 0,
    dpan_purchases_num_l1wk NUMBER DEFAULT 0, dpan_purchases_sum_l1wk FLOAT DEFAULT 0,
    dpan_distinct_merchants_l1wk NUMBER DEFAULT 0, dpan_declined_num_l1wk NUMBER DEFAULT 0
);

-- Entity 4: IP Velocity (Hybrid)
CREATE OR REPLACE HYBRID TABLE FRAUD_DEMO_DEV.FEATURES.IP_VELOCITY_RT (
    ip_address VARCHAR PRIMARY KEY,
    last_updated_ts TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    latest_txn_ts TIMESTAMP_NTZ,
    ip_purchases_num_l1h NUMBER DEFAULT 0, ip_distinct_customers_l1h NUMBER DEFAULT 0,
    ip_distinct_merchants_l1h NUMBER DEFAULT 0, ip_declined_num_l1h NUMBER DEFAULT 0,
    ip_purchases_num_l6h NUMBER DEFAULT 0, ip_distinct_customers_l6h NUMBER DEFAULT 0,
    ip_distinct_merchants_l6h NUMBER DEFAULT 0, ip_declined_num_l6h NUMBER DEFAULT 0,
    ip_purchases_num_l24h NUMBER DEFAULT 0, ip_distinct_customers_l24h NUMBER DEFAULT 0,
    ip_distinct_merchants_l24h NUMBER DEFAULT 0, ip_declined_num_l24h NUMBER DEFAULT 0,
    ip_purchases_num_l48h NUMBER DEFAULT 0, ip_distinct_customers_l48h NUMBER DEFAULT 0,
    ip_distinct_merchants_l48h NUMBER DEFAULT 0, ip_declined_num_l48h NUMBER DEFAULT 0,
    ip_purchases_num_l1wk NUMBER DEFAULT 0, ip_distinct_customers_l1wk NUMBER DEFAULT 0,
    ip_distinct_merchants_l1wk NUMBER DEFAULT 0, ip_declined_num_l1wk NUMBER DEFAULT 0
);

-- Entity 5: Customer-Merchant Velocity (Hybrid)
CREATE OR REPLACE HYBRID TABLE FRAUD_DEMO_DEV.FEATURES.CUSTOMER_MERCHANT_VELOCITY_RT (
    customer_id VARCHAR,
    merchant_id VARCHAR,
    last_updated_ts TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    latest_txn_ts TIMESTAMP_NTZ,
    cust_merch_purchases_num_l1h NUMBER DEFAULT 0, cust_merch_purchases_sum_l1h FLOAT DEFAULT 0,
    cust_merch_purchases_num_l6h NUMBER DEFAULT 0, cust_merch_purchases_sum_l6h FLOAT DEFAULT 0,
    cust_merch_purchases_num_l24h NUMBER DEFAULT 0, cust_merch_purchases_sum_l24h FLOAT DEFAULT 0,
    cust_merch_purchases_num_l48h NUMBER DEFAULT 0, cust_merch_purchases_sum_l48h FLOAT DEFAULT 0,
    cust_merch_purchases_num_l1wk NUMBER DEFAULT 0, cust_merch_purchases_sum_l1wk FLOAT DEFAULT 0,
    PRIMARY KEY (customer_id, merchant_id)
);

SELECT 'All 5 Hybrid Tables created. Fast task MERGEs directly into these.' AS status;

## 6.3 Create the Fast Task (10-second refresh)

This Task runs every 10 seconds (Snowflake's minimum task schedule). When the Stream has new data, it performs an incremental MERGE into all 5 entity feature tables via a stored procedure.

**Why 10 seconds?**
- **10 seconds is Snowflake's hard minimum** for task schedules. You cannot set a lower value.
- Each task execution has 2-5s of scheduling overhead (queue, check stream, compile, start). Even at 10s, the task uses about 30-50% of its interval on overhead.
- At 60 txn/min, each 10s cycle batches about 10 transactions per execution - efficient.

**Why not TARGET_LAG = DOWNSTREAM?**
- DOWNSTREAM means "only refresh when a downstream object requests it." Since these feature tables are leaf nodes (consumed directly by the scoring endpoint, not by another Dynamic Table), downstream lag would mean they never refresh proactively. Features would only update when a query happens to read them - defeating the purpose of real-time freshness.

**Why not 1-second refresh?**
- Snowflake does not support task schedules below 10 seconds. The CREATE TASK will fail with: `Cannot set schedule less than 10 seconds.`
- Even if it were supported, the 2-5s scheduling overhead per execution means tasks would constantly overlap or skip cycles at 1s intervals.
- For sub-second requirements, you would need a fundamentally different architecture (e.g., Kafka + Flink for stream processing outside Snowflake).

**What the fast task does:**
- Consumes new rows from the Stream (atomic - one consumption per execution)
- Stages them into a temporary table
- MERGEs into all 5 entity feature tables in sequence
- Updates ALL 5 time windows per entity (a new transaction falls within all windows simultaneously)

**What it does NOT do:**
- Expire old transactions that fall outside their respective windows (handled by the slow task)
- Compute DISTINCT counts accurately (incrementing a distinct count is not additive - the slow task corrects this)

In [ ]:
-- 6.3: Fast Task Stored Procedure + Task (10s, all 5 entities)
-- Note: 10 seconds is Snowflake's minimum task schedule.
CREATE OR REPLACE PROCEDURE FRAUD_DEMO_DEV.FEATURES.SP_FAST_FEATURE_UPDATE()
RETURNS STRING LANGUAGE SQL EXECUTE AS CALLER
AS $$
BEGIN
    CREATE OR REPLACE TEMPORARY TABLE FRAUD_DEMO_DEV.FEATURES._STREAM_BATCH AS
    SELECT * FROM FRAUD_DEMO_DEV.FEATURES.TXN_STREAM;

    -- Entity 1: CUSTOMER VELOCITY (all 5 windows incremented)
    MERGE INTO FRAUD_DEMO_DEV.FEATURES.CUSTOMER_VELOCITY_RT t
    USING (SELECT customer_id, COUNT(*) AS new_count, SUM(purchase_amount) AS new_sum, MIN(purchase_amount) AS new_min, MAX(purchase_amount) AS new_max, COUNT(DISTINCT purchase_amount) AS new_distinct_amt, SUM(CASE WHEN merchant_country = 'GBR' THEN 1 ELSE 0 END) AS new_gbr, SUM(CASE WHEN merchant_country != 'GBR' THEN 1 ELSE 0 END) AS new_nongbr, COUNT(DISTINCT card_token) AS new_tokens, COUNT(DISTINCT wallet_dpan) AS new_dpans, COUNT(DISTINCT merchant_id) AS new_merchants, COUNT(DISTINCT merchant_country) AS new_countries, SUM(CASE WHEN was_declined THEN 1 ELSE 0 END) AS new_declined, MAX(transaction_ts) AS latest_ts FROM FRAUD_DEMO_DEV.FEATURES._STREAM_BATCH GROUP BY customer_id) s
    ON t.customer_id = s.customer_id
    WHEN MATCHED THEN UPDATE SET t.purchases_num_l1h=t.purchases_num_l1h+s.new_count, t.purchases_sum_l1h=t.purchases_sum_l1h+s.new_sum, t.purchases_min_l1h=LEAST(COALESCE(t.purchases_min_l1h,s.new_min),s.new_min), t.purchases_max_l1h=GREATEST(t.purchases_max_l1h,s.new_max), t.distinct_purchase_amt_l1h=t.distinct_purchase_amt_l1h+s.new_distinct_amt, t.purchases_gbr_num_l1h=t.purchases_gbr_num_l1h+s.new_gbr, t.purchases_nongbr_num_l1h=t.purchases_nongbr_num_l1h+s.new_nongbr, t.distinct_card_tokens_l1h=t.distinct_card_tokens_l1h+s.new_tokens, t.distinct_wallet_dpan_l1h=t.distinct_wallet_dpan_l1h+s.new_dpans, t.distinct_merchants_l1h=t.distinct_merchants_l1h+s.new_merchants, t.declined_num_l1h=t.declined_num_l1h+s.new_declined, t.purchases_num_l6h=t.purchases_num_l6h+s.new_count, t.purchases_sum_l6h=t.purchases_sum_l6h+s.new_sum, t.purchases_min_l6h=LEAST(COALESCE(t.purchases_min_l6h,s.new_min),s.new_min), t.purchases_max_l6h=GREATEST(t.purchases_max_l6h,s.new_max), t.distinct_purchase_amt_l6h=t.distinct_purchase_amt_l6h+s.new_distinct_amt, t.purchases_gbr_num_l6h=t.purchases_gbr_num_l6h+s.new_gbr, t.purchases_nongbr_num_l6h=t.purchases_nongbr_num_l6h+s.new_nongbr, t.distinct_card_tokens_l6h=t.distinct_card_tokens_l6h+s.new_tokens, t.distinct_wallet_dpan_l6h=t.distinct_wallet_dpan_l6h+s.new_dpans, t.distinct_merchants_l6h=t.distinct_merchants_l6h+s.new_merchants, t.declined_num_l6h=t.declined_num_l6h+s.new_declined, t.purchases_num_l24h=t.purchases_num_l24h+s.new_count, t.purchases_sum_l24h=t.purchases_sum_l24h+s.new_sum, t.purchases_min_l24h=LEAST(COALESCE(t.purchases_min_l24h,s.new_min),s.new_min), t.purchases_max_l24h=GREATEST(t.purchases_max_l24h,s.new_max), t.distinct_purchase_amt_l24h=t.distinct_purchase_amt_l24h+s.new_distinct_amt, t.purchases_gbr_num_l24h=t.purchases_gbr_num_l24h+s.new_gbr, t.purchases_nongbr_num_l24h=t.purchases_nongbr_num_l24h+s.new_nongbr, t.distinct_card_tokens_l24h=t.distinct_card_tokens_l24h+s.new_tokens, t.distinct_wallet_dpan_l24h=t.distinct_wallet_dpan_l24h+s.new_dpans, t.distinct_merchants_l24h=t.distinct_merchants_l24h+s.new_merchants, t.distinct_countries_l24h=t.distinct_countries_l24h+s.new_countries, t.declined_num_l24h=t.declined_num_l24h+s.new_declined, t.purchases_num_l48h=t.purchases_num_l48h+s.new_count, t.purchases_sum_l48h=t.purchases_sum_l48h+s.new_sum, t.purchases_min_l48h=LEAST(COALESCE(t.purchases_min_l48h,s.new_min),s.new_min), t.purchases_max_l48h=GREATEST(t.purchases_max_l48h,s.new_max), t.distinct_purchase_amt_l48h=t.distinct_purchase_amt_l48h+s.new_distinct_amt, t.purchases_gbr_num_l48h=t.purchases_gbr_num_l48h+s.new_gbr, t.purchases_nongbr_num_l48h=t.purchases_nongbr_num_l48h+s.new_nongbr, t.distinct_card_tokens_l48h=t.distinct_card_tokens_l48h+s.new_tokens, t.distinct_wallet_dpan_l48h=t.distinct_wallet_dpan_l48h+s.new_dpans, t.distinct_merchants_l48h=t.distinct_merchants_l48h+s.new_merchants, t.declined_num_l48h=t.declined_num_l48h+s.new_declined, t.purchases_num_l1wk=t.purchases_num_l1wk+s.new_count, t.purchases_sum_l1wk=t.purchases_sum_l1wk+s.new_sum, t.purchases_min_l1wk=LEAST(COALESCE(t.purchases_min_l1wk,s.new_min),s.new_min), t.purchases_max_l1wk=GREATEST(t.purchases_max_l1wk,s.new_max), t.distinct_purchase_amt_l1wk=t.distinct_purchase_amt_l1wk+s.new_distinct_amt, t.purchases_gbr_num_l1wk=t.purchases_gbr_num_l1wk+s.new_gbr, t.purchases_nongbr_num_l1wk=t.purchases_nongbr_num_l1wk+s.new_nongbr, t.distinct_card_tokens_l1wk=t.distinct_card_tokens_l1wk+s.new_tokens, t.distinct_wallet_dpan_l1wk=t.distinct_wallet_dpan_l1wk+s.new_dpans, t.distinct_merchants_l1wk=t.distinct_merchants_l1wk+s.new_merchants, t.distinct_countries_l1wk=t.distinct_countries_l1wk+s.new_countries, t.declined_num_l1wk=t.declined_num_l1wk+s.new_declined, t.latest_txn_ts=s.latest_ts, t.last_updated_ts=CURRENT_TIMESTAMP()
    WHEN NOT MATCHED THEN INSERT (customer_id, latest_txn_ts, last_updated_ts, purchases_num_l1h, purchases_sum_l1h, purchases_min_l1h, purchases_max_l1h, distinct_purchase_amt_l1h, purchases_gbr_num_l1h, purchases_nongbr_num_l1h, distinct_card_tokens_l1h, distinct_wallet_dpan_l1h, distinct_merchants_l1h, declined_num_l1h, purchases_num_l6h, purchases_sum_l6h, purchases_min_l6h, purchases_max_l6h, distinct_purchase_amt_l6h, purchases_gbr_num_l6h, purchases_nongbr_num_l6h, distinct_card_tokens_l6h, distinct_wallet_dpan_l6h, distinct_merchants_l6h, declined_num_l6h, purchases_num_l24h, purchases_sum_l24h, purchases_min_l24h, purchases_max_l24h, distinct_purchase_amt_l24h, purchases_gbr_num_l24h, purchases_nongbr_num_l24h, distinct_card_tokens_l24h, distinct_wallet_dpan_l24h, distinct_merchants_l24h, distinct_countries_l24h, declined_num_l24h, purchases_num_l48h, purchases_sum_l48h, purchases_min_l48h, purchases_max_l48h, distinct_purchase_amt_l48h, purchases_gbr_num_l48h, purchases_nongbr_num_l48h, distinct_card_tokens_l48h, distinct_wallet_dpan_l48h, distinct_merchants_l48h, declined_num_l48h, purchases_num_l1wk, purchases_sum_l1wk, purchases_min_l1wk, purchases_max_l1wk, distinct_purchase_amt_l1wk, purchases_gbr_num_l1wk, purchases_nongbr_num_l1wk, distinct_card_tokens_l1wk, distinct_wallet_dpan_l1wk, distinct_merchants_l1wk, distinct_countries_l1wk, declined_num_l1wk) VALUES (s.customer_id, s.latest_ts, CURRENT_TIMESTAMP(), s.new_count, s.new_sum, s.new_min, s.new_max, s.new_distinct_amt, s.new_gbr, s.new_nongbr, s.new_tokens, s.new_dpans, s.new_merchants, s.new_declined, s.new_count, s.new_sum, s.new_min, s.new_max, s.new_distinct_amt, s.new_gbr, s.new_nongbr, s.new_tokens, s.new_dpans, s.new_merchants, s.new_declined, s.new_count, s.new_sum, s.new_min, s.new_max, s.new_distinct_amt, s.new_gbr, s.new_nongbr, s.new_tokens, s.new_dpans, s.new_merchants, s.new_countries, s.new_declined, s.new_count, s.new_sum, s.new_min, s.new_max, s.new_distinct_amt, s.new_gbr, s.new_nongbr, s.new_tokens, s.new_dpans, s.new_merchants, s.new_declined, s.new_count, s.new_sum, s.new_min, s.new_max, s.new_distinct_amt, s.new_gbr, s.new_nongbr, s.new_tokens, s.new_dpans, s.new_merchants, s.new_countries, s.new_declined);

    -- Entity 2: MERCHANT VELOCITY
    MERGE INTO FRAUD_DEMO_DEV.FEATURES.MERCHANT_VELOCITY_RT t USING (SELECT merchant_id, COUNT(*) AS n, SUM(purchase_amount) AS s_amt, COUNT(DISTINCT customer_id) AS dist_cust, SUM(CASE WHEN was_declined THEN 1 ELSE 0 END) AS decl, MAX(transaction_ts) AS latest_ts FROM FRAUD_DEMO_DEV.FEATURES._STREAM_BATCH GROUP BY merchant_id) s ON t.merchant_id = s.merchant_id WHEN MATCHED THEN UPDATE SET t.merchant_purchases_num_l1h=t.merchant_purchases_num_l1h+s.n, t.merchant_purchases_sum_l1h=t.merchant_purchases_sum_l1h+s.s_amt, t.merchant_distinct_customers_l1h=t.merchant_distinct_customers_l1h+s.dist_cust, t.merchant_declined_num_l1h=t.merchant_declined_num_l1h+s.decl, t.merchant_purchases_num_l6h=t.merchant_purchases_num_l6h+s.n, t.merchant_purchases_sum_l6h=t.merchant_purchases_sum_l6h+s.s_amt, t.merchant_distinct_customers_l6h=t.merchant_distinct_customers_l6h+s.dist_cust, t.merchant_declined_num_l6h=t.merchant_declined_num_l6h+s.decl, t.merchant_purchases_num_l24h=t.merchant_purchases_num_l24h+s.n, t.merchant_purchases_sum_l24h=t.merchant_purchases_sum_l24h+s.s_amt, t.merchant_distinct_customers_l24h=t.merchant_distinct_customers_l24h+s.dist_cust, t.merchant_declined_num_l24h=t.merchant_declined_num_l24h+s.decl, t.merchant_purchases_num_l48h=t.merchant_purchases_num_l48h+s.n, t.merchant_purchases_sum_l48h=t.merchant_purchases_sum_l48h+s.s_amt, t.merchant_distinct_customers_l48h=t.merchant_distinct_customers_l48h+s.dist_cust, t.merchant_declined_num_l48h=t.merchant_declined_num_l48h+s.decl, t.merchant_purchases_num_l1wk=t.merchant_purchases_num_l1wk+s.n, t.merchant_purchases_sum_l1wk=t.merchant_purchases_sum_l1wk+s.s_amt, t.merchant_distinct_customers_l1wk=t.merchant_distinct_customers_l1wk+s.dist_cust, t.merchant_declined_num_l1wk=t.merchant_declined_num_l1wk+s.decl, t.latest_txn_ts=s.latest_ts, t.last_updated_ts=CURRENT_TIMESTAMP() WHEN NOT MATCHED THEN INSERT (merchant_id, latest_txn_ts, last_updated_ts, merchant_purchases_num_l1h, merchant_purchases_sum_l1h, merchant_distinct_customers_l1h, merchant_declined_num_l1h, merchant_purchases_num_l6h, merchant_purchases_sum_l6h, merchant_distinct_customers_l6h, merchant_declined_num_l6h, merchant_purchases_num_l24h, merchant_purchases_sum_l24h, merchant_distinct_customers_l24h, merchant_declined_num_l24h, merchant_purchases_num_l48h, merchant_purchases_sum_l48h, merchant_distinct_customers_l48h, merchant_declined_num_l48h, merchant_purchases_num_l1wk, merchant_purchases_sum_l1wk, merchant_distinct_customers_l1wk, merchant_declined_num_l1wk) VALUES (s.merchant_id, s.latest_ts, CURRENT_TIMESTAMP(), s.n, s.s_amt, s.dist_cust, s.decl, s.n, s.s_amt, s.dist_cust, s.decl, s.n, s.s_amt, s.dist_cust, s.decl, s.n, s.s_amt, s.dist_cust, s.decl, s.n, s.s_amt, s.dist_cust, s.decl);

    -- Entity 3: WALLET DPAN VELOCITY
    MERGE INTO FRAUD_DEMO_DEV.FEATURES.WALLET_DPAN_VELOCITY_RT t USING (SELECT wallet_dpan, COUNT(*) AS n, SUM(purchase_amount) AS s_amt, COUNT(DISTINCT merchant_id) AS dist_merch, SUM(CASE WHEN was_declined THEN 1 ELSE 0 END) AS decl, MAX(transaction_ts) AS latest_ts FROM FRAUD_DEMO_DEV.FEATURES._STREAM_BATCH GROUP BY wallet_dpan) s ON t.wallet_dpan = s.wallet_dpan WHEN MATCHED THEN UPDATE SET t.dpan_purchases_num_l1h=t.dpan_purchases_num_l1h+s.n, t.dpan_purchases_sum_l1h=t.dpan_purchases_sum_l1h+s.s_amt, t.dpan_distinct_merchants_l1h=t.dpan_distinct_merchants_l1h+s.dist_merch, t.dpan_declined_num_l1h=t.dpan_declined_num_l1h+s.decl, t.dpan_purchases_num_l6h=t.dpan_purchases_num_l6h+s.n, t.dpan_purchases_sum_l6h=t.dpan_purchases_sum_l6h+s.s_amt, t.dpan_distinct_merchants_l6h=t.dpan_distinct_merchants_l6h+s.dist_merch, t.dpan_declined_num_l6h=t.dpan_declined_num_l6h+s.decl, t.dpan_purchases_num_l24h=t.dpan_purchases_num_l24h+s.n, t.dpan_purchases_sum_l24h=t.dpan_purchases_sum_l24h+s.s_amt, t.dpan_distinct_merchants_l24h=t.dpan_distinct_merchants_l24h+s.dist_merch, t.dpan_declined_num_l24h=t.dpan_declined_num_l24h+s.decl, t.dpan_purchases_num_l48h=t.dpan_purchases_num_l48h+s.n, t.dpan_purchases_sum_l48h=t.dpan_purchases_sum_l48h+s.s_amt, t.dpan_distinct_merchants_l48h=t.dpan_distinct_merchants_l48h+s.dist_merch, t.dpan_declined_num_l48h=t.dpan_declined_num_l48h+s.decl, t.dpan_purchases_num_l1wk=t.dpan_purchases_num_l1wk+s.n, t.dpan_purchases_sum_l1wk=t.dpan_purchases_sum_l1wk+s.s_amt, t.dpan_distinct_merchants_l1wk=t.dpan_distinct_merchants_l1wk+s.dist_merch, t.dpan_declined_num_l1wk=t.dpan_declined_num_l1wk+s.decl, t.latest_txn_ts=s.latest_ts, t.last_updated_ts=CURRENT_TIMESTAMP() WHEN NOT MATCHED THEN INSERT (wallet_dpan, latest_txn_ts, last_updated_ts, dpan_purchases_num_l1h, dpan_purchases_sum_l1h, dpan_distinct_merchants_l1h, dpan_declined_num_l1h, dpan_purchases_num_l6h, dpan_purchases_sum_l6h, dpan_distinct_merchants_l6h, dpan_declined_num_l6h, dpan_purchases_num_l24h, dpan_purchases_sum_l24h, dpan_distinct_merchants_l24h, dpan_declined_num_l24h, dpan_purchases_num_l48h, dpan_purchases_sum_l48h, dpan_distinct_merchants_l48h, dpan_declined_num_l48h, dpan_purchases_num_l1wk, dpan_purchases_sum_l1wk, dpan_distinct_merchants_l1wk, dpan_declined_num_l1wk) VALUES (s.wallet_dpan, s.latest_ts, CURRENT_TIMESTAMP(), s.n, s.s_amt, s.dist_merch, s.decl, s.n, s.s_amt, s.dist_merch, s.decl, s.n, s.s_amt, s.dist_merch, s.decl, s.n, s.s_amt, s.dist_merch, s.decl, s.n, s.s_amt, s.dist_merch, s.decl);

    -- Entity 4: IP VELOCITY
    MERGE INTO FRAUD_DEMO_DEV.FEATURES.IP_VELOCITY_RT t USING (SELECT ip_address, COUNT(*) AS n, COUNT(DISTINCT customer_id) AS dist_cust, COUNT(DISTINCT merchant_id) AS dist_merch, SUM(CASE WHEN was_declined THEN 1 ELSE 0 END) AS decl, MAX(transaction_ts) AS latest_ts FROM FRAUD_DEMO_DEV.FEATURES._STREAM_BATCH GROUP BY ip_address) s ON t.ip_address = s.ip_address WHEN MATCHED THEN UPDATE SET t.ip_purchases_num_l1h=t.ip_purchases_num_l1h+s.n, t.ip_distinct_customers_l1h=t.ip_distinct_customers_l1h+s.dist_cust, t.ip_distinct_merchants_l1h=t.ip_distinct_merchants_l1h+s.dist_merch, t.ip_declined_num_l1h=t.ip_declined_num_l1h+s.decl, t.ip_purchases_num_l6h=t.ip_purchases_num_l6h+s.n, t.ip_distinct_customers_l6h=t.ip_distinct_customers_l6h+s.dist_cust, t.ip_distinct_merchants_l6h=t.ip_distinct_merchants_l6h+s.dist_merch, t.ip_declined_num_l6h=t.ip_declined_num_l6h+s.decl, t.ip_purchases_num_l24h=t.ip_purchases_num_l24h+s.n, t.ip_distinct_customers_l24h=t.ip_distinct_customers_l24h+s.dist_cust, t.ip_distinct_merchants_l24h=t.ip_distinct_merchants_l24h+s.dist_merch, t.ip_declined_num_l24h=t.ip_declined_num_l24h+s.decl, t.ip_purchases_num_l48h=t.ip_purchases_num_l48h+s.n, t.ip_distinct_customers_l48h=t.ip_distinct_customers_l48h+s.dist_cust, t.ip_distinct_merchants_l48h=t.ip_distinct_merchants_l48h+s.dist_merch, t.ip_declined_num_l48h=t.ip_declined_num_l48h+s.decl, t.ip_purchases_num_l1wk=t.ip_purchases_num_l1wk+s.n, t.ip_distinct_customers_l1wk=t.ip_distinct_customers_l1wk+s.dist_cust, t.ip_distinct_merchants_l1wk=t.ip_distinct_merchants_l1wk+s.dist_merch, t.ip_declined_num_l1wk=t.ip_declined_num_l1wk+s.decl, t.latest_txn_ts=s.latest_ts, t.last_updated_ts=CURRENT_TIMESTAMP() WHEN NOT MATCHED THEN INSERT (ip_address, latest_txn_ts, last_updated_ts, ip_purchases_num_l1h, ip_distinct_customers_l1h, ip_distinct_merchants_l1h, ip_declined_num_l1h, ip_purchases_num_l6h, ip_distinct_customers_l6h, ip_distinct_merchants_l6h, ip_declined_num_l6h, ip_purchases_num_l24h, ip_distinct_customers_l24h, ip_distinct_merchants_l24h, ip_declined_num_l24h, ip_purchases_num_l48h, ip_distinct_customers_l48h, ip_distinct_merchants_l48h, ip_declined_num_l48h, ip_purchases_num_l1wk, ip_distinct_customers_l1wk, ip_distinct_merchants_l1wk, ip_declined_num_l1wk) VALUES (s.ip_address, s.latest_ts, CURRENT_TIMESTAMP(), s.n, s.dist_cust, s.dist_merch, s.decl, s.n, s.dist_cust, s.dist_merch, s.decl, s.n, s.dist_cust, s.dist_merch, s.decl, s.n, s.dist_cust, s.dist_merch, s.decl, s.n, s.dist_cust, s.dist_merch, s.decl);

    -- Entity 5: CUSTOMER-MERCHANT VELOCITY
    MERGE INTO FRAUD_DEMO_DEV.FEATURES.CUSTOMER_MERCHANT_VELOCITY_RT t USING (SELECT customer_id, merchant_id, COUNT(*) AS n, SUM(purchase_amount) AS s_amt, MAX(transaction_ts) AS latest_ts FROM FRAUD_DEMO_DEV.FEATURES._STREAM_BATCH GROUP BY customer_id, merchant_id) s ON t.customer_id = s.customer_id AND t.merchant_id = s.merchant_id WHEN MATCHED THEN UPDATE SET t.cust_merch_purchases_num_l1h=t.cust_merch_purchases_num_l1h+s.n, t.cust_merch_purchases_sum_l1h=t.cust_merch_purchases_sum_l1h+s.s_amt, t.cust_merch_purchases_num_l6h=t.cust_merch_purchases_num_l6h+s.n, t.cust_merch_purchases_sum_l6h=t.cust_merch_purchases_sum_l6h+s.s_amt, t.cust_merch_purchases_num_l24h=t.cust_merch_purchases_num_l24h+s.n, t.cust_merch_purchases_sum_l24h=t.cust_merch_purchases_sum_l24h+s.s_amt, t.cust_merch_purchases_num_l48h=t.cust_merch_purchases_num_l48h+s.n, t.cust_merch_purchases_sum_l48h=t.cust_merch_purchases_sum_l48h+s.s_amt, t.cust_merch_purchases_num_l1wk=t.cust_merch_purchases_num_l1wk+s.n, t.cust_merch_purchases_sum_l1wk=t.cust_merch_purchases_sum_l1wk+s.s_amt, t.latest_txn_ts=s.latest_ts, t.last_updated_ts=CURRENT_TIMESTAMP() WHEN NOT MATCHED THEN INSERT (customer_id, merchant_id, latest_txn_ts, last_updated_ts, cust_merch_purchases_num_l1h, cust_merch_purchases_sum_l1h, cust_merch_purchases_num_l6h, cust_merch_purchases_sum_l6h, cust_merch_purchases_num_l24h, cust_merch_purchases_sum_l24h, cust_merch_purchases_num_l48h, cust_merch_purchases_sum_l48h, cust_merch_purchases_num_l1wk, cust_merch_purchases_sum_l1wk) VALUES (s.customer_id, s.merchant_id, s.latest_ts, CURRENT_TIMESTAMP(), s.n, s.s_amt, s.n, s.s_amt, s.n, s.s_amt, s.n, s.s_amt, s.n, s.s_amt);

    DROP TABLE IF EXISTS FRAUD_DEMO_DEV.FEATURES._STREAM_BATCH;
    RETURN '5 entity tables updated';
END;
$$;

CREATE OR REPLACE TASK FRAUD_DEMO_DEV.FEATURES.FAST_FEATURE_UPDATE
    WAREHOUSE = FRAUD_DEMO_WH
    SCHEDULE = '10 SECOND'
    WHEN SYSTEM$STREAM_HAS_DATA('FRAUD_DEMO_DEV.FEATURES.TXN_STREAM')
AS CALL FRAUD_DEMO_DEV.FEATURES.SP_FAST_FEATURE_UPDATE();

ALTER TASK FRAUD_DEMO_DEV.FEATURES.FAST_FEATURE_UPDATE RESUME;
SELECT 'Fast task created and resumed (all 5 entities, every 10s)' AS status;

## 6.4 Create the Slow Task (Window Expiry - 5 min, all 5 entities)

This Task handles the **window expiry problem** for all 5 Hybrid Tables. Every 5 minutes, it fully recomputes the rolling window aggregates from the source transaction table and MERGEs into the **same Hybrid Tables** that the fast task writes to.

This corrects two types of drift from the incremental fast task:
1. **Expired transactions** - rows that aged out of their window (e.g., a purchase from 61 minutes ago should no longer count in the 1-hour window)
2. **DISTINCT count over-estimation** - the fast task increments distinct counts even when the value was already seen in the window

After this runs, all features are **exactly correct** (identical to what the Dynamic Table in NB02 would produce).

### How Both Tasks Share the Same Tables

```
                  HYBRID TABLES (created in 6.2)
                  [CUSTOMER_VELOCITY_RT, MERCHANT_VELOCITY_RT, ...]
                         /                    \
                        /                      \
    Fast Task (10s)    /                        \   Slow Task (5 min)
    - Incremental     /                          \  - Full recompute
    - Adds new txns  /                            \ - Corrects drift
    - Approximate   /                              \ - Exact values
                   /                                \
              WRITES                              WRITES
              (MERGE)                             (MERGE)
                         \
                          \--- Scoring Service READS (sub-10ms point lookup)
                               (NB04 section 4.4)
```

There is **one set of tables** serving three purposes:
- Fast task writes incrementally (10s freshness for new activity)
- Slow task overwrites with exact values (corrects drift every 5 min)
- Scoring service reads by primary key (sub-10ms at transaction time)

In [ ]:
-- 6.4: Slow Task - full window recompute every 5 minutes (all 5 entities)
CREATE OR REPLACE PROCEDURE FRAUD_DEMO_DEV.FEATURES.SP_WINDOW_EXPIRY_RECOMPUTE()
RETURNS STRING LANGUAGE SQL EXECUTE AS CALLER
AS $$
BEGIN
    -- Entity 1: CUSTOMER VELOCITY (full recompute, all 5 windows)
    MERGE INTO FRAUD_DEMO_DEV.FEATURES.CUSTOMER_VELOCITY_RT t
    USING (
        SELECT customer_id, MAX(transaction_ts) AS latest_txn_ts,
            COUNT(CASE WHEN transaction_ts > DATEADD('hour',-1,CURRENT_TIMESTAMP()) THEN 1 END) AS purchases_num_l1h, COALESCE(SUM(CASE WHEN transaction_ts > DATEADD('hour',-1,CURRENT_TIMESTAMP()) THEN purchase_amount END),0) AS purchases_sum_l1h, MIN(CASE WHEN transaction_ts > DATEADD('hour',-1,CURRENT_TIMESTAMP()) THEN purchase_amount END) AS purchases_min_l1h, MAX(CASE WHEN transaction_ts > DATEADD('hour',-1,CURRENT_TIMESTAMP()) THEN purchase_amount END) AS purchases_max_l1h, COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour',-1,CURRENT_TIMESTAMP()) THEN purchase_amount END) AS distinct_purchase_amt_l1h, COUNT(CASE WHEN transaction_ts > DATEADD('hour',-1,CURRENT_TIMESTAMP()) AND merchant_country='GBR' THEN 1 END) AS purchases_gbr_num_l1h, COUNT(CASE WHEN transaction_ts > DATEADD('hour',-1,CURRENT_TIMESTAMP()) AND merchant_country!='GBR' THEN 1 END) AS purchases_nongbr_num_l1h, COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour',-1,CURRENT_TIMESTAMP()) THEN card_token END) AS distinct_card_tokens_l1h, COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour',-1,CURRENT_TIMESTAMP()) THEN wallet_dpan END) AS distinct_wallet_dpan_l1h, COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour',-1,CURRENT_TIMESTAMP()) THEN merchant_id END) AS distinct_merchants_l1h, COUNT(CASE WHEN transaction_ts > DATEADD('hour',-1,CURRENT_TIMESTAMP()) AND was_declined THEN 1 END) AS declined_num_l1h,
            COUNT(CASE WHEN transaction_ts > DATEADD('hour',-6,CURRENT_TIMESTAMP()) THEN 1 END) AS purchases_num_l6h, COALESCE(SUM(CASE WHEN transaction_ts > DATEADD('hour',-6,CURRENT_TIMESTAMP()) THEN purchase_amount END),0) AS purchases_sum_l6h, MIN(CASE WHEN transaction_ts > DATEADD('hour',-6,CURRENT_TIMESTAMP()) THEN purchase_amount END) AS purchases_min_l6h, MAX(CASE WHEN transaction_ts > DATEADD('hour',-6,CURRENT_TIMESTAMP()) THEN purchase_amount END) AS purchases_max_l6h, COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour',-6,CURRENT_TIMESTAMP()) THEN purchase_amount END) AS distinct_purchase_amt_l6h, COUNT(CASE WHEN transaction_ts > DATEADD('hour',-6,CURRENT_TIMESTAMP()) AND merchant_country='GBR' THEN 1 END) AS purchases_gbr_num_l6h, COUNT(CASE WHEN transaction_ts > DATEADD('hour',-6,CURRENT_TIMESTAMP()) AND merchant_country!='GBR' THEN 1 END) AS purchases_nongbr_num_l6h, COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour',-6,CURRENT_TIMESTAMP()) THEN card_token END) AS distinct_card_tokens_l6h, COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour',-6,CURRENT_TIMESTAMP()) THEN wallet_dpan END) AS distinct_wallet_dpan_l6h, COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour',-6,CURRENT_TIMESTAMP()) THEN merchant_id END) AS distinct_merchants_l6h, COUNT(CASE WHEN transaction_ts > DATEADD('hour',-6,CURRENT_TIMESTAMP()) AND was_declined THEN 1 END) AS declined_num_l6h,
            COUNT(CASE WHEN transaction_ts > DATEADD('hour',-24,CURRENT_TIMESTAMP()) THEN 1 END) AS purchases_num_l24h, COALESCE(SUM(CASE WHEN transaction_ts > DATEADD('hour',-24,CURRENT_TIMESTAMP()) THEN purchase_amount END),0) AS purchases_sum_l24h, MIN(CASE WHEN transaction_ts > DATEADD('hour',-24,CURRENT_TIMESTAMP()) THEN purchase_amount END) AS purchases_min_l24h, MAX(CASE WHEN transaction_ts > DATEADD('hour',-24,CURRENT_TIMESTAMP()) THEN purchase_amount END) AS purchases_max_l24h, COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour',-24,CURRENT_TIMESTAMP()) THEN purchase_amount END) AS distinct_purchase_amt_l24h, COUNT(CASE WHEN transaction_ts > DATEADD('hour',-24,CURRENT_TIMESTAMP()) AND merchant_country='GBR' THEN 1 END) AS purchases_gbr_num_l24h, COUNT(CASE WHEN transaction_ts > DATEADD('hour',-24,CURRENT_TIMESTAMP()) AND merchant_country!='GBR' THEN 1 END) AS purchases_nongbr_num_l24h, COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour',-24,CURRENT_TIMESTAMP()) THEN card_token END) AS distinct_card_tokens_l24h, COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour',-24,CURRENT_TIMESTAMP()) THEN wallet_dpan END) AS distinct_wallet_dpan_l24h, COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour',-24,CURRENT_TIMESTAMP()) THEN merchant_id END) AS distinct_merchants_l24h, COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour',-24,CURRENT_TIMESTAMP()) THEN merchant_country END) AS distinct_countries_l24h, COUNT(CASE WHEN transaction_ts > DATEADD('hour',-24,CURRENT_TIMESTAMP()) AND was_declined THEN 1 END) AS declined_num_l24h,
            COUNT(CASE WHEN transaction_ts > DATEADD('hour',-48,CURRENT_TIMESTAMP()) THEN 1 END) AS purchases_num_l48h, COALESCE(SUM(CASE WHEN transaction_ts > DATEADD('hour',-48,CURRENT_TIMESTAMP()) THEN purchase_amount END),0) AS purchases_sum_l48h, MIN(CASE WHEN transaction_ts > DATEADD('hour',-48,CURRENT_TIMESTAMP()) THEN purchase_amount END) AS purchases_min_l48h, MAX(CASE WHEN transaction_ts > DATEADD('hour',-48,CURRENT_TIMESTAMP()) THEN purchase_amount END) AS purchases_max_l48h, COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour',-48,CURRENT_TIMESTAMP()) THEN purchase_amount END) AS distinct_purchase_amt_l48h, COUNT(CASE WHEN transaction_ts > DATEADD('hour',-48,CURRENT_TIMESTAMP()) AND merchant_country='GBR' THEN 1 END) AS purchases_gbr_num_l48h, COUNT(CASE WHEN transaction_ts > DATEADD('hour',-48,CURRENT_TIMESTAMP()) AND merchant_country!='GBR' THEN 1 END) AS purchases_nongbr_num_l48h, COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour',-48,CURRENT_TIMESTAMP()) THEN card_token END) AS distinct_card_tokens_l48h, COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour',-48,CURRENT_TIMESTAMP()) THEN wallet_dpan END) AS distinct_wallet_dpan_l48h, COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour',-48,CURRENT_TIMESTAMP()) THEN merchant_id END) AS distinct_merchants_l48h, COUNT(CASE WHEN transaction_ts > DATEADD('hour',-48,CURRENT_TIMESTAMP()) AND was_declined THEN 1 END) AS declined_num_l48h,
            COUNT(CASE WHEN transaction_ts > DATEADD('day',-7,CURRENT_TIMESTAMP()) THEN 1 END) AS purchases_num_l1wk, COALESCE(SUM(CASE WHEN transaction_ts > DATEADD('day',-7,CURRENT_TIMESTAMP()) THEN purchase_amount END),0) AS purchases_sum_l1wk, MIN(CASE WHEN transaction_ts > DATEADD('day',-7,CURRENT_TIMESTAMP()) THEN purchase_amount END) AS purchases_min_l1wk, MAX(CASE WHEN transaction_ts > DATEADD('day',-7,CURRENT_TIMESTAMP()) THEN purchase_amount END) AS purchases_max_l1wk, COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('day',-7,CURRENT_TIMESTAMP()) THEN purchase_amount END) AS distinct_purchase_amt_l1wk, COUNT(CASE WHEN transaction_ts > DATEADD('day',-7,CURRENT_TIMESTAMP()) AND merchant_country='GBR' THEN 1 END) AS purchases_gbr_num_l1wk, COUNT(CASE WHEN transaction_ts > DATEADD('day',-7,CURRENT_TIMESTAMP()) AND merchant_country!='GBR' THEN 1 END) AS purchases_nongbr_num_l1wk, COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('day',-7,CURRENT_TIMESTAMP()) THEN card_token END) AS distinct_card_tokens_l1wk, COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('day',-7,CURRENT_TIMESTAMP()) THEN wallet_dpan END) AS distinct_wallet_dpan_l1wk, COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('day',-7,CURRENT_TIMESTAMP()) THEN merchant_id END) AS distinct_merchants_l1wk, COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('day',-7,CURRENT_TIMESTAMP()) THEN merchant_country END) AS distinct_countries_l1wk, COUNT(CASE WHEN transaction_ts > DATEADD('day',-7,CURRENT_TIMESTAMP()) AND was_declined THEN 1 END) AS declined_num_l1wk
        FROM FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS WHERE transaction_ts > DATEADD('day',-7,CURRENT_TIMESTAMP()) GROUP BY customer_id
    ) s ON t.customer_id = s.customer_id
    WHEN MATCHED THEN UPDATE SET t.latest_txn_ts=s.latest_txn_ts, t.last_updated_ts=CURRENT_TIMESTAMP(), t.purchases_num_l1h=s.purchases_num_l1h, t.purchases_sum_l1h=s.purchases_sum_l1h, t.purchases_min_l1h=s.purchases_min_l1h, t.purchases_max_l1h=s.purchases_max_l1h, t.distinct_purchase_amt_l1h=s.distinct_purchase_amt_l1h, t.purchases_gbr_num_l1h=s.purchases_gbr_num_l1h, t.purchases_nongbr_num_l1h=s.purchases_nongbr_num_l1h, t.distinct_card_tokens_l1h=s.distinct_card_tokens_l1h, t.distinct_wallet_dpan_l1h=s.distinct_wallet_dpan_l1h, t.distinct_merchants_l1h=s.distinct_merchants_l1h, t.declined_num_l1h=s.declined_num_l1h, t.purchases_num_l6h=s.purchases_num_l6h, t.purchases_sum_l6h=s.purchases_sum_l6h, t.purchases_min_l6h=s.purchases_min_l6h, t.purchases_max_l6h=s.purchases_max_l6h, t.distinct_purchase_amt_l6h=s.distinct_purchase_amt_l6h, t.purchases_gbr_num_l6h=s.purchases_gbr_num_l6h, t.purchases_nongbr_num_l6h=s.purchases_nongbr_num_l6h, t.distinct_card_tokens_l6h=s.distinct_card_tokens_l6h, t.distinct_wallet_dpan_l6h=s.distinct_wallet_dpan_l6h, t.distinct_merchants_l6h=s.distinct_merchants_l6h, t.declined_num_l6h=s.declined_num_l6h, t.purchases_num_l24h=s.purchases_num_l24h, t.purchases_sum_l24h=s.purchases_sum_l24h, t.purchases_min_l24h=s.purchases_min_l24h, t.purchases_max_l24h=s.purchases_max_l24h, t.distinct_purchase_amt_l24h=s.distinct_purchase_amt_l24h, t.purchases_gbr_num_l24h=s.purchases_gbr_num_l24h, t.purchases_nongbr_num_l24h=s.purchases_nongbr_num_l24h, t.distinct_card_tokens_l24h=s.distinct_card_tokens_l24h, t.distinct_wallet_dpan_l24h=s.distinct_wallet_dpan_l24h, t.distinct_merchants_l24h=s.distinct_merchants_l24h, t.distinct_countries_l24h=s.distinct_countries_l24h, t.declined_num_l24h=s.declined_num_l24h, t.purchases_num_l48h=s.purchases_num_l48h, t.purchases_sum_l48h=s.purchases_sum_l48h, t.purchases_min_l48h=s.purchases_min_l48h, t.purchases_max_l48h=s.purchases_max_l48h, t.distinct_purchase_amt_l48h=s.distinct_purchase_amt_l48h, t.purchases_gbr_num_l48h=s.purchases_gbr_num_l48h, t.purchases_nongbr_num_l48h=s.purchases_nongbr_num_l48h, t.distinct_card_tokens_l48h=s.distinct_card_tokens_l48h, t.distinct_wallet_dpan_l48h=s.distinct_wallet_dpan_l48h, t.distinct_merchants_l48h=s.distinct_merchants_l48h, t.declined_num_l48h=s.declined_num_l48h, t.purchases_num_l1wk=s.purchases_num_l1wk, t.purchases_sum_l1wk=s.purchases_sum_l1wk, t.purchases_min_l1wk=s.purchases_min_l1wk, t.purchases_max_l1wk=s.purchases_max_l1wk, t.distinct_purchase_amt_l1wk=s.distinct_purchase_amt_l1wk, t.purchases_gbr_num_l1wk=s.purchases_gbr_num_l1wk, t.purchases_nongbr_num_l1wk=s.purchases_nongbr_num_l1wk, t.distinct_card_tokens_l1wk=s.distinct_card_tokens_l1wk, t.distinct_wallet_dpan_l1wk=s.distinct_wallet_dpan_l1wk, t.distinct_merchants_l1wk=s.distinct_merchants_l1wk, t.distinct_countries_l1wk=s.distinct_countries_l1wk, t.declined_num_l1wk=s.declined_num_l1wk
    WHEN NOT MATCHED THEN INSERT (customer_id, latest_txn_ts, last_updated_ts, purchases_num_l1h, purchases_sum_l1h, purchases_min_l1h, purchases_max_l1h, distinct_purchase_amt_l1h, purchases_gbr_num_l1h, purchases_nongbr_num_l1h, distinct_card_tokens_l1h, distinct_wallet_dpan_l1h, distinct_merchants_l1h, declined_num_l1h, purchases_num_l6h, purchases_sum_l6h, purchases_min_l6h, purchases_max_l6h, distinct_purchase_amt_l6h, purchases_gbr_num_l6h, purchases_nongbr_num_l6h, distinct_card_tokens_l6h, distinct_wallet_dpan_l6h, distinct_merchants_l6h, declined_num_l6h, purchases_num_l24h, purchases_sum_l24h, purchases_min_l24h, purchases_max_l24h, distinct_purchase_amt_l24h, purchases_gbr_num_l24h, purchases_nongbr_num_l24h, distinct_card_tokens_l24h, distinct_wallet_dpan_l24h, distinct_merchants_l24h, distinct_countries_l24h, declined_num_l24h, purchases_num_l48h, purchases_sum_l48h, purchases_min_l48h, purchases_max_l48h, distinct_purchase_amt_l48h, purchases_gbr_num_l48h, purchases_nongbr_num_l48h, distinct_card_tokens_l48h, distinct_wallet_dpan_l48h, distinct_merchants_l48h, declined_num_l48h, purchases_num_l1wk, purchases_sum_l1wk, purchases_min_l1wk, purchases_max_l1wk, distinct_purchase_amt_l1wk, purchases_gbr_num_l1wk, purchases_nongbr_num_l1wk, distinct_card_tokens_l1wk, distinct_wallet_dpan_l1wk, distinct_merchants_l1wk, distinct_countries_l1wk, declined_num_l1wk) VALUES (s.customer_id, s.latest_txn_ts, CURRENT_TIMESTAMP(), s.purchases_num_l1h, s.purchases_sum_l1h, s.purchases_min_l1h, s.purchases_max_l1h, s.distinct_purchase_amt_l1h, s.purchases_gbr_num_l1h, s.purchases_nongbr_num_l1h, s.distinct_card_tokens_l1h, s.distinct_wallet_dpan_l1h, s.distinct_merchants_l1h, s.declined_num_l1h, s.purchases_num_l6h, s.purchases_sum_l6h, s.purchases_min_l6h, s.purchases_max_l6h, s.distinct_purchase_amt_l6h, s.purchases_gbr_num_l6h, s.purchases_nongbr_num_l6h, s.distinct_card_tokens_l6h, s.distinct_wallet_dpan_l6h, s.distinct_merchants_l6h, s.declined_num_l6h, s.purchases_num_l24h, s.purchases_sum_l24h, s.purchases_min_l24h, s.purchases_max_l24h, s.distinct_purchase_amt_l24h, s.purchases_gbr_num_l24h, s.purchases_nongbr_num_l24h, s.distinct_card_tokens_l24h, s.distinct_wallet_dpan_l24h, s.distinct_merchants_l24h, s.distinct_countries_l24h, s.declined_num_l24h, s.purchases_num_l48h, s.purchases_sum_l48h, s.purchases_min_l48h, s.purchases_max_l48h, s.distinct_purchase_amt_l48h, s.purchases_gbr_num_l48h, s.purchases_nongbr_num_l48h, s.distinct_card_tokens_l48h, s.distinct_wallet_dpan_l48h, s.distinct_merchants_l48h, s.declined_num_l48h, s.purchases_num_l1wk, s.purchases_sum_l1wk, s.purchases_min_l1wk, s.purchases_max_l1wk, s.distinct_purchase_amt_l1wk, s.purchases_gbr_num_l1wk, s.purchases_nongbr_num_l1wk, s.distinct_card_tokens_l1wk, s.distinct_wallet_dpan_l1wk, s.distinct_merchants_l1wk, s.distinct_countries_l1wk, s.declined_num_l1wk);

    -- Entity 2: MERCHANT VELOCITY (full recompute)
    MERGE INTO FRAUD_DEMO_DEV.FEATURES.MERCHANT_VELOCITY_RT t USING (SELECT merchant_id, MAX(transaction_ts) AS latest_txn_ts, COUNT(CASE WHEN transaction_ts > DATEADD('hour',-1,CURRENT_TIMESTAMP()) THEN 1 END) AS merchant_purchases_num_l1h, COALESCE(SUM(CASE WHEN transaction_ts > DATEADD('hour',-1,CURRENT_TIMESTAMP()) THEN purchase_amount END),0) AS merchant_purchases_sum_l1h, COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour',-1,CURRENT_TIMESTAMP()) THEN customer_id END) AS merchant_distinct_customers_l1h, COUNT(CASE WHEN transaction_ts > DATEADD('hour',-1,CURRENT_TIMESTAMP()) AND was_declined THEN 1 END) AS merchant_declined_num_l1h, COUNT(CASE WHEN transaction_ts > DATEADD('hour',-6,CURRENT_TIMESTAMP()) THEN 1 END) AS merchant_purchases_num_l6h, COALESCE(SUM(CASE WHEN transaction_ts > DATEADD('hour',-6,CURRENT_TIMESTAMP()) THEN purchase_amount END),0) AS merchant_purchases_sum_l6h, COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour',-6,CURRENT_TIMESTAMP()) THEN customer_id END) AS merchant_distinct_customers_l6h, COUNT(CASE WHEN transaction_ts > DATEADD('hour',-6,CURRENT_TIMESTAMP()) AND was_declined THEN 1 END) AS merchant_declined_num_l6h, COUNT(CASE WHEN transaction_ts > DATEADD('hour',-24,CURRENT_TIMESTAMP()) THEN 1 END) AS merchant_purchases_num_l24h, COALESCE(SUM(CASE WHEN transaction_ts > DATEADD('hour',-24,CURRENT_TIMESTAMP()) THEN purchase_amount END),0) AS merchant_purchases_sum_l24h, COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour',-24,CURRENT_TIMESTAMP()) THEN customer_id END) AS merchant_distinct_customers_l24h, COUNT(CASE WHEN transaction_ts > DATEADD('hour',-24,CURRENT_TIMESTAMP()) AND was_declined THEN 1 END) AS merchant_declined_num_l24h, COUNT(CASE WHEN transaction_ts > DATEADD('hour',-48,CURRENT_TIMESTAMP()) THEN 1 END) AS merchant_purchases_num_l48h, COALESCE(SUM(CASE WHEN transaction_ts > DATEADD('hour',-48,CURRENT_TIMESTAMP()) THEN purchase_amount END),0) AS merchant_purchases_sum_l48h, COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour',-48,CURRENT_TIMESTAMP()) THEN customer_id END) AS merchant_distinct_customers_l48h, COUNT(CASE WHEN transaction_ts > DATEADD('hour',-48,CURRENT_TIMESTAMP()) AND was_declined THEN 1 END) AS merchant_declined_num_l48h, COUNT(CASE WHEN transaction_ts > DATEADD('day',-7,CURRENT_TIMESTAMP()) THEN 1 END) AS merchant_purchases_num_l1wk, COALESCE(SUM(CASE WHEN transaction_ts > DATEADD('day',-7,CURRENT_TIMESTAMP()) THEN purchase_amount END),0) AS merchant_purchases_sum_l1wk, COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('day',-7,CURRENT_TIMESTAMP()) THEN customer_id END) AS merchant_distinct_customers_l1wk, COUNT(CASE WHEN transaction_ts > DATEADD('day',-7,CURRENT_TIMESTAMP()) AND was_declined THEN 1 END) AS merchant_declined_num_l1wk FROM FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS WHERE transaction_ts > DATEADD('day',-7,CURRENT_TIMESTAMP()) GROUP BY merchant_id) s ON t.merchant_id = s.merchant_id WHEN MATCHED THEN UPDATE SET t.latest_txn_ts=s.latest_txn_ts, t.last_updated_ts=CURRENT_TIMESTAMP(), t.merchant_purchases_num_l1h=s.merchant_purchases_num_l1h, t.merchant_purchases_sum_l1h=s.merchant_purchases_sum_l1h, t.merchant_distinct_customers_l1h=s.merchant_distinct_customers_l1h, t.merchant_declined_num_l1h=s.merchant_declined_num_l1h, t.merchant_purchases_num_l6h=s.merchant_purchases_num_l6h, t.merchant_purchases_sum_l6h=s.merchant_purchases_sum_l6h, t.merchant_distinct_customers_l6h=s.merchant_distinct_customers_l6h, t.merchant_declined_num_l6h=s.merchant_declined_num_l6h, t.merchant_purchases_num_l24h=s.merchant_purchases_num_l24h, t.merchant_purchases_sum_l24h=s.merchant_purchases_sum_l24h, t.merchant_distinct_customers_l24h=s.merchant_distinct_customers_l24h, t.merchant_declined_num_l24h=s.merchant_declined_num_l24h, t.merchant_purchases_num_l48h=s.merchant_purchases_num_l48h, t.merchant_purchases_sum_l48h=s.merchant_purchases_sum_l48h, t.merchant_distinct_customers_l48h=s.merchant_distinct_customers_l48h, t.merchant_declined_num_l48h=s.merchant_declined_num_l48h, t.merchant_purchases_num_l1wk=s.merchant_purchases_num_l1wk, t.merchant_purchases_sum_l1wk=s.merchant_purchases_sum_l1wk, t.merchant_distinct_customers_l1wk=s.merchant_distinct_customers_l1wk, t.merchant_declined_num_l1wk=s.merchant_declined_num_l1wk WHEN NOT MATCHED THEN INSERT (merchant_id, latest_txn_ts, last_updated_ts, merchant_purchases_num_l1h, merchant_purchases_sum_l1h, merchant_distinct_customers_l1h, merchant_declined_num_l1h, merchant_purchases_num_l6h, merchant_purchases_sum_l6h, merchant_distinct_customers_l6h, merchant_declined_num_l6h, merchant_purchases_num_l24h, merchant_purchases_sum_l24h, merchant_distinct_customers_l24h, merchant_declined_num_l24h, merchant_purchases_num_l48h, merchant_purchases_sum_l48h, merchant_distinct_customers_l48h, merchant_declined_num_l48h, merchant_purchases_num_l1wk, merchant_purchases_sum_l1wk, merchant_distinct_customers_l1wk, merchant_declined_num_l1wk) VALUES (s.merchant_id, s.latest_txn_ts, CURRENT_TIMESTAMP(), s.merchant_purchases_num_l1h, s.merchant_purchases_sum_l1h, s.merchant_distinct_customers_l1h, s.merchant_declined_num_l1h, s.merchant_purchases_num_l6h, s.merchant_purchases_sum_l6h, s.merchant_distinct_customers_l6h, s.merchant_declined_num_l6h, s.merchant_purchases_num_l24h, s.merchant_purchases_sum_l24h, s.merchant_distinct_customers_l24h, s.merchant_declined_num_l24h, s.merchant_purchases_num_l48h, s.merchant_purchases_sum_l48h, s.merchant_distinct_customers_l48h, s.merchant_declined_num_l48h, s.merchant_purchases_num_l1wk, s.merchant_purchases_sum_l1wk, s.merchant_distinct_customers_l1wk, s.merchant_declined_num_l1wk);

    -- Entity 3: WALLET DPAN VELOCITY (full recompute)
    MERGE INTO FRAUD_DEMO_DEV.FEATURES.WALLET_DPAN_VELOCITY_RT t USING (SELECT wallet_dpan, MAX(transaction_ts) AS latest_txn_ts, COUNT(CASE WHEN transaction_ts > DATEADD('hour',-1,CURRENT_TIMESTAMP()) THEN 1 END) AS dpan_purchases_num_l1h, COALESCE(SUM(CASE WHEN transaction_ts > DATEADD('hour',-1,CURRENT_TIMESTAMP()) THEN purchase_amount END),0) AS dpan_purchases_sum_l1h, COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour',-1,CURRENT_TIMESTAMP()) THEN merchant_id END) AS dpan_distinct_merchants_l1h, COUNT(CASE WHEN transaction_ts > DATEADD('hour',-1,CURRENT_TIMESTAMP()) AND was_declined THEN 1 END) AS dpan_declined_num_l1h, COUNT(CASE WHEN transaction_ts > DATEADD('hour',-6,CURRENT_TIMESTAMP()) THEN 1 END) AS dpan_purchases_num_l6h, COALESCE(SUM(CASE WHEN transaction_ts > DATEADD('hour',-6,CURRENT_TIMESTAMP()) THEN purchase_amount END),0) AS dpan_purchases_sum_l6h, COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour',-6,CURRENT_TIMESTAMP()) THEN merchant_id END) AS dpan_distinct_merchants_l6h, COUNT(CASE WHEN transaction_ts > DATEADD('hour',-6,CURRENT_TIMESTAMP()) AND was_declined THEN 1 END) AS dpan_declined_num_l6h, COUNT(CASE WHEN transaction_ts > DATEADD('hour',-24,CURRENT_TIMESTAMP()) THEN 1 END) AS dpan_purchases_num_l24h, COALESCE(SUM(CASE WHEN transaction_ts > DATEADD('hour',-24,CURRENT_TIMESTAMP()) THEN purchase_amount END),0) AS dpan_purchases_sum_l24h, COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour',-24,CURRENT_TIMESTAMP()) THEN merchant_id END) AS dpan_distinct_merchants_l24h, COUNT(CASE WHEN transaction_ts > DATEADD('hour',-24,CURRENT_TIMESTAMP()) AND was_declined THEN 1 END) AS dpan_declined_num_l24h, COUNT(CASE WHEN transaction_ts > DATEADD('hour',-48,CURRENT_TIMESTAMP()) THEN 1 END) AS dpan_purchases_num_l48h, COALESCE(SUM(CASE WHEN transaction_ts > DATEADD('hour',-48,CURRENT_TIMESTAMP()) THEN purchase_amount END),0) AS dpan_purchases_sum_l48h, COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour',-48,CURRENT_TIMESTAMP()) THEN merchant_id END) AS dpan_distinct_merchants_l48h, COUNT(CASE WHEN transaction_ts > DATEADD('hour',-48,CURRENT_TIMESTAMP()) AND was_declined THEN 1 END) AS dpan_declined_num_l48h, COUNT(CASE WHEN transaction_ts > DATEADD('day',-7,CURRENT_TIMESTAMP()) THEN 1 END) AS dpan_purchases_num_l1wk, COALESCE(SUM(CASE WHEN transaction_ts > DATEADD('day',-7,CURRENT_TIMESTAMP()) THEN purchase_amount END),0) AS dpan_purchases_sum_l1wk, COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('day',-7,CURRENT_TIMESTAMP()) THEN merchant_id END) AS dpan_distinct_merchants_l1wk, COUNT(CASE WHEN transaction_ts > DATEADD('day',-7,CURRENT_TIMESTAMP()) AND was_declined THEN 1 END) AS dpan_declined_num_l1wk FROM FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS WHERE transaction_ts > DATEADD('day',-7,CURRENT_TIMESTAMP()) GROUP BY wallet_dpan) s ON t.wallet_dpan = s.wallet_dpan WHEN MATCHED THEN UPDATE SET t.latest_txn_ts=s.latest_txn_ts, t.last_updated_ts=CURRENT_TIMESTAMP(), t.dpan_purchases_num_l1h=s.dpan_purchases_num_l1h, t.dpan_purchases_sum_l1h=s.dpan_purchases_sum_l1h, t.dpan_distinct_merchants_l1h=s.dpan_distinct_merchants_l1h, t.dpan_declined_num_l1h=s.dpan_declined_num_l1h, t.dpan_purchases_num_l6h=s.dpan_purchases_num_l6h, t.dpan_purchases_sum_l6h=s.dpan_purchases_sum_l6h, t.dpan_distinct_merchants_l6h=s.dpan_distinct_merchants_l6h, t.dpan_declined_num_l6h=s.dpan_declined_num_l6h, t.dpan_purchases_num_l24h=s.dpan_purchases_num_l24h, t.dpan_purchases_sum_l24h=s.dpan_purchases_sum_l24h, t.dpan_distinct_merchants_l24h=s.dpan_distinct_merchants_l24h, t.dpan_declined_num_l24h=s.dpan_declined_num_l24h, t.dpan_purchases_num_l48h=s.dpan_purchases_num_l48h, t.dpan_purchases_sum_l48h=s.dpan_purchases_sum_l48h, t.dpan_distinct_merchants_l48h=s.dpan_distinct_merchants_l48h, t.dpan_declined_num_l48h=s.dpan_declined_num_l48h, t.dpan_purchases_num_l1wk=s.dpan_purchases_num_l1wk, t.dpan_purchases_sum_l1wk=s.dpan_purchases_sum_l1wk, t.dpan_distinct_merchants_l1wk=s.dpan_distinct_merchants_l1wk, t.dpan_declined_num_l1wk=s.dpan_declined_num_l1wk WHEN NOT MATCHED THEN INSERT (wallet_dpan, latest_txn_ts, last_updated_ts, dpan_purchases_num_l1h, dpan_purchases_sum_l1h, dpan_distinct_merchants_l1h, dpan_declined_num_l1h, dpan_purchases_num_l6h, dpan_purchases_sum_l6h, dpan_distinct_merchants_l6h, dpan_declined_num_l6h, dpan_purchases_num_l24h, dpan_purchases_sum_l24h, dpan_distinct_merchants_l24h, dpan_declined_num_l24h, dpan_purchases_num_l48h, dpan_purchases_sum_l48h, dpan_distinct_merchants_l48h, dpan_declined_num_l48h, dpan_purchases_num_l1wk, dpan_purchases_sum_l1wk, dpan_distinct_merchants_l1wk, dpan_declined_num_l1wk) VALUES (s.wallet_dpan, s.latest_txn_ts, CURRENT_TIMESTAMP(), s.dpan_purchases_num_l1h, s.dpan_purchases_sum_l1h, s.dpan_distinct_merchants_l1h, s.dpan_declined_num_l1h, s.dpan_purchases_num_l6h, s.dpan_purchases_sum_l6h, s.dpan_distinct_merchants_l6h, s.dpan_declined_num_l6h, s.dpan_purchases_num_l24h, s.dpan_purchases_sum_l24h, s.dpan_distinct_merchants_l24h, s.dpan_declined_num_l24h, s.dpan_purchases_num_l48h, s.dpan_purchases_sum_l48h, s.dpan_distinct_merchants_l48h, s.dpan_declined_num_l48h, s.dpan_purchases_num_l1wk, s.dpan_purchases_sum_l1wk, s.dpan_distinct_merchants_l1wk, s.dpan_declined_num_l1wk);

    -- Entity 4: IP VELOCITY (full recompute)
    MERGE INTO FRAUD_DEMO_DEV.FEATURES.IP_VELOCITY_RT t USING (SELECT ip_address, MAX(transaction_ts) AS latest_txn_ts, COUNT(CASE WHEN transaction_ts > DATEADD('hour',-1,CURRENT_TIMESTAMP()) THEN 1 END) AS ip_purchases_num_l1h, COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour',-1,CURRENT_TIMESTAMP()) THEN customer_id END) AS ip_distinct_customers_l1h, COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour',-1,CURRENT_TIMESTAMP()) THEN merchant_id END) AS ip_distinct_merchants_l1h, COUNT(CASE WHEN transaction_ts > DATEADD('hour',-1,CURRENT_TIMESTAMP()) AND was_declined THEN 1 END) AS ip_declined_num_l1h, COUNT(CASE WHEN transaction_ts > DATEADD('hour',-6,CURRENT_TIMESTAMP()) THEN 1 END) AS ip_purchases_num_l6h, COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour',-6,CURRENT_TIMESTAMP()) THEN customer_id END) AS ip_distinct_customers_l6h, COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour',-6,CURRENT_TIMESTAMP()) THEN merchant_id END) AS ip_distinct_merchants_l6h, COUNT(CASE WHEN transaction_ts > DATEADD('hour',-6,CURRENT_TIMESTAMP()) AND was_declined THEN 1 END) AS ip_declined_num_l6h, COUNT(CASE WHEN transaction_ts > DATEADD('hour',-24,CURRENT_TIMESTAMP()) THEN 1 END) AS ip_purchases_num_l24h, COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour',-24,CURRENT_TIMESTAMP()) THEN customer_id END) AS ip_distinct_customers_l24h, COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour',-24,CURRENT_TIMESTAMP()) THEN merchant_id END) AS ip_distinct_merchants_l24h, COUNT(CASE WHEN transaction_ts > DATEADD('hour',-24,CURRENT_TIMESTAMP()) AND was_declined THEN 1 END) AS ip_declined_num_l24h, COUNT(CASE WHEN transaction_ts > DATEADD('hour',-48,CURRENT_TIMESTAMP()) THEN 1 END) AS ip_purchases_num_l48h, COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour',-48,CURRENT_TIMESTAMP()) THEN customer_id END) AS ip_distinct_customers_l48h, COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('hour',-48,CURRENT_TIMESTAMP()) THEN merchant_id END) AS ip_distinct_merchants_l48h, COUNT(CASE WHEN transaction_ts > DATEADD('hour',-48,CURRENT_TIMESTAMP()) AND was_declined THEN 1 END) AS ip_declined_num_l48h, COUNT(CASE WHEN transaction_ts > DATEADD('day',-7,CURRENT_TIMESTAMP()) THEN 1 END) AS ip_purchases_num_l1wk, COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('day',-7,CURRENT_TIMESTAMP()) THEN customer_id END) AS ip_distinct_customers_l1wk, COUNT(DISTINCT CASE WHEN transaction_ts > DATEADD('day',-7,CURRENT_TIMESTAMP()) THEN merchant_id END) AS ip_distinct_merchants_l1wk, COUNT(CASE WHEN transaction_ts > DATEADD('day',-7,CURRENT_TIMESTAMP()) AND was_declined THEN 1 END) AS ip_declined_num_l1wk FROM FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS WHERE transaction_ts > DATEADD('day',-7,CURRENT_TIMESTAMP()) GROUP BY ip_address) s ON t.ip_address = s.ip_address WHEN MATCHED THEN UPDATE SET t.latest_txn_ts=s.latest_txn_ts, t.last_updated_ts=CURRENT_TIMESTAMP(), t.ip_purchases_num_l1h=s.ip_purchases_num_l1h, t.ip_distinct_customers_l1h=s.ip_distinct_customers_l1h, t.ip_distinct_merchants_l1h=s.ip_distinct_merchants_l1h, t.ip_declined_num_l1h=s.ip_declined_num_l1h, t.ip_purchases_num_l6h=s.ip_purchases_num_l6h, t.ip_distinct_customers_l6h=s.ip_distinct_customers_l6h, t.ip_distinct_merchants_l6h=s.ip_distinct_merchants_l6h, t.ip_declined_num_l6h=s.ip_declined_num_l6h, t.ip_purchases_num_l24h=s.ip_purchases_num_l24h, t.ip_distinct_customers_l24h=s.ip_distinct_customers_l24h, t.ip_distinct_merchants_l24h=s.ip_distinct_merchants_l24h, t.ip_declined_num_l24h=s.ip_declined_num_l24h, t.ip_purchases_num_l48h=s.ip_purchases_num_l48h, t.ip_distinct_customers_l48h=s.ip_distinct_customers_l48h, t.ip_distinct_merchants_l48h=s.ip_distinct_merchants_l48h, t.ip_declined_num_l48h=s.ip_declined_num_l48h, t.ip_purchases_num_l1wk=s.ip_purchases_num_l1wk, t.ip_distinct_customers_l1wk=s.ip_distinct_customers_l1wk, t.ip_distinct_merchants_l1wk=s.ip_distinct_merchants_l1wk, t.ip_declined_num_l1wk=s.ip_declined_num_l1wk WHEN NOT MATCHED THEN INSERT (ip_address, latest_txn_ts, last_updated_ts, ip_purchases_num_l1h, ip_distinct_customers_l1h, ip_distinct_merchants_l1h, ip_declined_num_l1h, ip_purchases_num_l6h, ip_distinct_customers_l6h, ip_distinct_merchants_l6h, ip_declined_num_l6h, ip_purchases_num_l24h, ip_distinct_customers_l24h, ip_distinct_merchants_l24h, ip_declined_num_l24h, ip_purchases_num_l48h, ip_distinct_customers_l48h, ip_distinct_merchants_l48h, ip_declined_num_l48h, ip_purchases_num_l1wk, ip_distinct_customers_l1wk, ip_distinct_merchants_l1wk, ip_declined_num_l1wk) VALUES (s.ip_address, s.latest_txn_ts, CURRENT_TIMESTAMP(), s.ip_purchases_num_l1h, s.ip_distinct_customers_l1h, s.ip_distinct_merchants_l1h, s.ip_declined_num_l1h, s.ip_purchases_num_l6h, s.ip_distinct_customers_l6h, s.ip_distinct_merchants_l6h, s.ip_declined_num_l6h, s.ip_purchases_num_l24h, s.ip_distinct_customers_l24h, s.ip_distinct_merchants_l24h, s.ip_declined_num_l24h, s.ip_purchases_num_l48h, s.ip_distinct_customers_l48h, s.ip_distinct_merchants_l48h, s.ip_declined_num_l48h, s.ip_purchases_num_l1wk, s.ip_distinct_customers_l1wk, s.ip_distinct_merchants_l1wk, s.ip_declined_num_l1wk);

    -- Entity 5: CUSTOMER-MERCHANT VELOCITY (full recompute)
    MERGE INTO FRAUD_DEMO_DEV.FEATURES.CUSTOMER_MERCHANT_VELOCITY_RT t USING (SELECT customer_id, merchant_id, MAX(transaction_ts) AS latest_txn_ts, COUNT(CASE WHEN transaction_ts > DATEADD('hour',-1,CURRENT_TIMESTAMP()) THEN 1 END) AS cust_merch_purchases_num_l1h, COALESCE(SUM(CASE WHEN transaction_ts > DATEADD('hour',-1,CURRENT_TIMESTAMP()) THEN purchase_amount END),0) AS cust_merch_purchases_sum_l1h, COUNT(CASE WHEN transaction_ts > DATEADD('hour',-6,CURRENT_TIMESTAMP()) THEN 1 END) AS cust_merch_purchases_num_l6h, COALESCE(SUM(CASE WHEN transaction_ts > DATEADD('hour',-6,CURRENT_TIMESTAMP()) THEN purchase_amount END),0) AS cust_merch_purchases_sum_l6h, COUNT(CASE WHEN transaction_ts > DATEADD('hour',-24,CURRENT_TIMESTAMP()) THEN 1 END) AS cust_merch_purchases_num_l24h, COALESCE(SUM(CASE WHEN transaction_ts > DATEADD('hour',-24,CURRENT_TIMESTAMP()) THEN purchase_amount END),0) AS cust_merch_purchases_sum_l24h, COUNT(CASE WHEN transaction_ts > DATEADD('hour',-48,CURRENT_TIMESTAMP()) THEN 1 END) AS cust_merch_purchases_num_l48h, COALESCE(SUM(CASE WHEN transaction_ts > DATEADD('hour',-48,CURRENT_TIMESTAMP()) THEN purchase_amount END),0) AS cust_merch_purchases_sum_l48h, COUNT(CASE WHEN transaction_ts > DATEADD('day',-7,CURRENT_TIMESTAMP()) THEN 1 END) AS cust_merch_purchases_num_l1wk, COALESCE(SUM(CASE WHEN transaction_ts > DATEADD('day',-7,CURRENT_TIMESTAMP()) THEN purchase_amount END),0) AS cust_merch_purchases_sum_l1wk FROM FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS WHERE transaction_ts > DATEADD('day',-7,CURRENT_TIMESTAMP()) GROUP BY customer_id, merchant_id) s ON t.customer_id = s.customer_id AND t.merchant_id = s.merchant_id WHEN MATCHED THEN UPDATE SET t.latest_txn_ts=s.latest_txn_ts, t.last_updated_ts=CURRENT_TIMESTAMP(), t.cust_merch_purchases_num_l1h=s.cust_merch_purchases_num_l1h, t.cust_merch_purchases_sum_l1h=s.cust_merch_purchases_sum_l1h, t.cust_merch_purchases_num_l6h=s.cust_merch_purchases_num_l6h, t.cust_merch_purchases_sum_l6h=s.cust_merch_purchases_sum_l6h, t.cust_merch_purchases_num_l24h=s.cust_merch_purchases_num_l24h, t.cust_merch_purchases_sum_l24h=s.cust_merch_purchases_sum_l24h, t.cust_merch_purchases_num_l48h=s.cust_merch_purchases_num_l48h, t.cust_merch_purchases_sum_l48h=s.cust_merch_purchases_sum_l48h, t.cust_merch_purchases_num_l1wk=s.cust_merch_purchases_num_l1wk, t.cust_merch_purchases_sum_l1wk=s.cust_merch_purchases_sum_l1wk WHEN NOT MATCHED THEN INSERT (customer_id, merchant_id, latest_txn_ts, last_updated_ts, cust_merch_purchases_num_l1h, cust_merch_purchases_sum_l1h, cust_merch_purchases_num_l6h, cust_merch_purchases_sum_l6h, cust_merch_purchases_num_l24h, cust_merch_purchases_sum_l24h, cust_merch_purchases_num_l48h, cust_merch_purchases_sum_l48h, cust_merch_purchases_num_l1wk, cust_merch_purchases_sum_l1wk) VALUES (s.customer_id, s.merchant_id, s.latest_txn_ts, CURRENT_TIMESTAMP(), s.cust_merch_purchases_num_l1h, s.cust_merch_purchases_sum_l1h, s.cust_merch_purchases_num_l6h, s.cust_merch_purchases_sum_l6h, s.cust_merch_purchases_num_l24h, s.cust_merch_purchases_sum_l24h, s.cust_merch_purchases_num_l48h, s.cust_merch_purchases_sum_l48h, s.cust_merch_purchases_num_l1wk, s.cust_merch_purchases_sum_l1wk);

    RETURN '5 entity tables recomputed (window expiry corrected)';
END;
$$;

CREATE OR REPLACE TASK FRAUD_DEMO_DEV.FEATURES.WINDOW_EXPIRY_RECOMPUTE
    WAREHOUSE = FRAUD_DEMO_WH
    SCHEDULE = '5 MINUTE'
AS CALL FRAUD_DEMO_DEV.FEATURES.SP_WINDOW_EXPIRY_RECOMPUTE();

ALTER TASK FRAUD_DEMO_DEV.FEATURES.WINDOW_EXPIRY_RECOMPUTE RESUME;
SELECT 'Slow task created and resumed (all 5 entities, every 5 min)' AS status;

## 6.5 Freshness Benchmark (RUN LIVE)

Same methodology as NB02: insert transactions, poll the feature table, measure time to reflect new data.

In [ ]:
import time
from datetime import datetime, timezone

run_id = datetime.now(timezone.utc).strftime('%Y%m%d%H%M%S')
test_customer = f'CUST-STREAM-{run_id}'
test_merchant = 'MERCH-000433'
test_dpan = 'DPAN-000000434'
test_ip = 'IP-0000432'
num_txns = 10

# Record baselines for shared entities (they may already have data)
baselines = {}
for name, table, key_col, key_val, count_col in [
    ('MERCHANT', 'FRAUD_DEMO_DEV.FEATURES.MERCHANT_VELOCITY_RT', 'merchant_id', test_merchant, 'MERCHANT_PURCHASES_NUM_L1H'),
    ('DPAN', 'FRAUD_DEMO_DEV.FEATURES.WALLET_DPAN_VELOCITY_RT', 'wallet_dpan', test_dpan, 'DPAN_PURCHASES_NUM_L1H'),
    ('IP', 'FRAUD_DEMO_DEV.FEATURES.IP_VELOCITY_RT', 'ip_address', test_ip, 'IP_PURCHASES_NUM_L1H'),
]:
    result = session.sql(f"SELECT {count_col} FROM {table} WHERE {key_col} = '{key_val}'").collect()
    baselines[name] = result[0][count_col] if result else 0

print(f"Inserting {num_txns} transactions...")
print(f"  Customer:  {test_customer} (unique, expect 0 -> {num_txns})")
print(f"  Merchant:  {test_merchant} (shared, baseline={baselines['MERCHANT']})")
print(f"  DPAN:      {test_dpan} (shared, baseline={baselines['DPAN']})")
print(f"  IP:        {test_ip} (shared, baseline={baselines['IP']})")

session.sql(f"""
    INSERT INTO FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS
    SELECT
        'TXN-STREAM-{run_id}-' || LPAD(SEQ4()::STRING, 4, '0') AS transaction_id,
        CURRENT_TIMESTAMP()::TIMESTAMP_NTZ AS transaction_ts,
        '{test_customer}' AS customer_id,
        '{test_merchant}' AS merchant_id,
        '{test_dpan}' AS wallet_dpan,
        '{test_ip}' AS ip_address,
        'TOKEN-00000001' AS card_token,
        ROUND(UNIFORM(10::FLOAT, 100::FLOAT, RANDOM()), 2) AS purchase_amount,
        'GBP' AS local_currency_code,
        'GBR' AS merchant_country,
        '5411' AS mcc_code,
        'NFC' AS tap_and_pay_type,
        'IPHONE' AS wallet_device_type,
        'APPLE_PAY' AS wallet_name,
        FALSE AS authenticated_3ds_challenge_flag,
        FALSE AS is_merchant_initiated_purchase,
        FALSE AS was_declined,
        FALSE AS is_fraud
    FROM TABLE(GENERATOR(ROWCOUNT => {num_txns}))
""").collect()

insert_time = time.time()
print(f"\nINSERT committed at {datetime.now(timezone.utc).strftime('%H:%M:%S')} UTC")
print(f"Polling all 5 entity tables...\n")

# Track detection times for all 5 entities
dt_checks = {
    'CUSTOMER_VELOCITY': {
        'table': 'FRAUD_DEMO_DEV.FEATURES.CUSTOMER_VELOCITY_RT',
        'key_col': 'customer_id', 'key_val': test_customer,
        'count_col': 'PURCHASES_NUM_L1H',
        'baseline': 0, 'detected_at': None
    },
    'MERCHANT_VELOCITY': {
        'table': 'FRAUD_DEMO_DEV.FEATURES.MERCHANT_VELOCITY_RT',
        'key_col': 'merchant_id', 'key_val': test_merchant,
        'count_col': 'MERCHANT_PURCHASES_NUM_L1H',
        'baseline': baselines['MERCHANT'], 'detected_at': None
    },
    'WALLET_DPAN_VELOCITY': {
        'table': 'FRAUD_DEMO_DEV.FEATURES.WALLET_DPAN_VELOCITY_RT',
        'key_col': 'wallet_dpan', 'key_val': test_dpan,
        'count_col': 'DPAN_PURCHASES_NUM_L1H',
        'baseline': baselines['DPAN'], 'detected_at': None
    },
    'IP_VELOCITY': {
        'table': 'FRAUD_DEMO_DEV.FEATURES.IP_VELOCITY_RT',
        'key_col': 'ip_address', 'key_val': test_ip,
        'count_col': 'IP_PURCHASES_NUM_L1H',
        'baseline': baselines['IP'], 'detected_at': None
    },
    'CUST_MERCH_VELOCITY': {
        'table': 'FRAUD_DEMO_DEV.FEATURES.CUSTOMER_MERCHANT_VELOCITY_RT',
        'key_col': 'customer_id', 'key_val': test_customer,
        'count_col': 'CUST_MERCH_PURCHASES_NUM_L1H',
        'baseline': 0, 'detected_at': None,
        'extra_filter': f"AND merchant_id = '{test_merchant}'"
    }
}

max_wait = 60
poll_interval = 1

while time.time() - insert_time < max_wait:
    pending = []
    for name, cfg in dt_checks.items():
        if cfg['detected_at'] is not None:
            continue

        extra = cfg.get('extra_filter', '')
        result = session.sql(f"""
            SELECT {cfg['count_col']}
            FROM {cfg['table']}
            WHERE {cfg['key_col']} = '{cfg['key_val']}' {extra}
        """).collect()

        current = result[0][cfg['count_col']] if result else 0
        expected = cfg['baseline'] + num_txns

        if current >= expected:
            cfg['detected_at'] = time.time() - insert_time
        else:
            pending.append(name)

    if not pending:
        break

    elapsed = time.time() - insert_time
    if int(elapsed) % 5 == 0 and elapsed > 1:
        print(f"  ... {elapsed:.0f}s | waiting on: {', '.join(pending)}")
    time.sleep(poll_interval)

# Results
print(f"\n{'='*65}")
print(f"STREAMS + TASKS FRESHNESS RESULTS (all 5 entities)")
print(f"{'='*65}")
print(f"{'Entity':<30} {'Latency':<12} {'Status'}")
print(f"{'-'*65}")

latencies = []
for name, cfg in dt_checks.items():
    if cfg['detected_at'] is not None:
        print(f"{name:<30} {cfg['detected_at']:.1f}s{'':>8} OK")
        latencies.append(cfg['detected_at'])
    else:
        print(f"{name:<30} {'>'}{max_wait}s{'':>7} TIMEOUT")

if latencies:
    print(f"\n  Fastest:      {min(latencies):.1f}s")
    print(f"  Slowest:      {max(latencies):.1f}s")
    print(f"  All detected: {len(latencies)}/5")
    print(f"\n  vs Dynamic Table (NB02):  33-39 seconds")
    print(f"  Improvement:              {39 / max(latencies):.1f}x faster")

    if max(latencies) <= 10:
        print(f"\n  VERDICT: Sub-10s SLA MET")
    elif max(latencies) <= 15:
        print(f"\n  VERDICT: Sub-15s achieved (close to sub-10s target)")
    else:
        print(f"\n  VERDICT: {max(latencies):.0f}s - not meeting sub-10s SLA")
else:
    print(f"\n  No entities detected. Check tasks are RESUMED.")

## 6.6 Cost Estimate

The Streams + Tasks approach has a similar warehouse cost profile to Dynamic Tables, with one key difference: the WHEN SYSTEM$STREAM_HAS_DATA clause means the fast task only runs when data exists.

However, at 60 txn/min, data is always flowing during business hours, so the warehouse stays warm either way.

In [ ]:
%%sql -r cost_comparison
-- Cost comparison: Streams + Tasks vs Dynamic Tables
-- Both use FRAUD_DEMO_WH (MEDIUM, 4 credits/hr)
-- Credit price: $4.58

SELECT
    approach,
    warehouse_size,
    credits_per_hr,
    always_on,
    credits_per_day,
    cost_per_month_usd,
    feature_freshness,
    complexity
FROM (
    SELECT
        'Dynamic Tables (NB02)' AS approach,
        'MEDIUM' AS warehouse_size,
        4 AS credits_per_hr,
        'YES (refresh interval < auto_suspend)' AS always_on,
        96 AS credits_per_day,
        ROUND(96 * 30 * 4.58, 0) AS cost_per_month_usd,
        '33-60 seconds' AS feature_freshness,
        'Low (declarative)' AS complexity

    UNION ALL

    SELECT
        'Streams + Tasks (this notebook)',
        'MEDIUM',
        4,
        'YES (task fires every 10s during business hours)',
        96,
        ROUND(96 * 30 * 4.58, 0),
        '5-15 seconds',
        'High (imperative MERGE + expiry logic)'
);

## 6.7 Challenges and Risks

### 1. Window Expiry Accuracy

The fast task only ADDS to counters. Between slow task runs (every 5 min), the 1-hour window features are **slightly inflated** because expired transactions haven't been subtracted yet.

**Impact on fraud detection:** A slightly inflated velocity count means the model may over-estimate activity, leading to marginally more false positives (blocking legitimate transactions). This is the safer direction for fraud - better to over-block than under-block.

### 2. Operational Complexity (5x Multiplied)

With Dynamic Tables, you write a SELECT and Snowflake handles everything. With Streams + Tasks for all 5 entities:
- **2 stored procedures** + **2 tasks** to monitor and maintain
- The stored procedures contain MERGE logic for all 5 entities (bugs in one = incorrect features)
- You must handle task failures (TASK_HISTORY monitoring)
- Schema changes require updating both stored procedures
- Adding a new feature column requires modifying both fast and slow procedures

### 3. Stream Consumption is Transactional

When the fast task consumes from the Stream, those rows are removed from the Stream. If the stored procedure fails mid-execution:
- The stream data has been consumed but the later MERGEs may not have completed
- The slow task will correct this within 5 minutes (full recompute)
- But during that window, some entities may have stale data

### 4. Hybrid Table Considerations

The `_RT` tables are Hybrid Tables (created in section 6.2). Key implications:
- **No CTAS** - cannot use CREATE TABLE AS SELECT; must INSERT/MERGE row-by-row
- **Storage model** - row-based, so large analytical scans are slower than standard tables
- **Cost** - Hybrid Tables have their own storage pricing (included in Enterprise Edition)
- **Availability** - requires Enterprise Edition or higher
- **Constraints enforced** - PRIMARY KEY violations will cause MERGE failures

### 5. Serving Architecture (Single Source of Truth)

The Hybrid Tables serve **dual purpose**:
- **Write path:** Fast task (10s) and slow task (5 min) MERGE into them
- **Read path:** Scoring service queries them by primary key at transaction time (NB04, section 4.4)

This eliminates data duplication and ensures the scoring service always reads the latest available features. There is no sync lag between a "feature store" and a "serving store" - they are the same table.

**End-to-end latency (from NB04 benchmark):**
- 5x Hybrid Table lookups: 25-50ms
- Model scoring (SPCS internal DNS): 100ms
- Total: 125-150ms (add 50-100ms for public ingress)

**Feature freshness:** 10-15 seconds (fast task schedule + execution time)

## 6.8 Hybrid Tables for Production Scoring

In production, the fast task should write directly to Hybrid Tables (not standard tables). This eliminates the extra hop of copying data from `_RT` tables to Hybrid Tables.

Below we create the Hybrid Table versions. In a production deployment, you would modify the stored procedure `SP_FAST_FEATURE_UPDATE` to MERGE into these Hybrid Tables directly instead of the standard `_RT` tables.

In [ ]:
%%sql -r cleanup_result
-- Suspend tasks to stop compute consumption
ALTER TASK FRAUD_DEMO_DEV.FEATURES.FAST_FEATURE_UPDATE SUSPEND;
ALTER TASK FRAUD_DEMO_DEV.FEATURES.WINDOW_EXPIRY_RECOMPUTE SUSPEND;

SELECT 'Tasks suspended. No further credits will be consumed.' AS status;

---
## Recommendation

### When to Use Each Approach

| Requirement | Recommendation | Why |
|---|---|---|
| Feature freshness 30-60s is acceptable | **Dynamic Tables (NB02)** | Zero maintenance, proven, same cost |
| Need 10-15s freshness + sub-200ms scoring | **Streams + Tasks + Hybrid Tables (this notebook)** | Fastest Snowflake-native path |
| Need sub-50ms scoring + 10-15s freshness | **Streams + Tasks + Hybrid Tables + SageMaker** | Hybrid lookup + external model |

### Measured SLAs

| Metric | Dynamic Tables (NB02) | Streams + Tasks (this notebook) |
|---|---|---|
| Feature freshness | 33-39 seconds | 10-15 seconds |
| Feature lookup latency | 200-500ms (standard table, warehouse) | 5-10ms (Hybrid Table, index) |
| End-to-end scoring (internal) | 105ms (model only) | 125-150ms (5 lookups + model) |
| End-to-end scoring (production est.) | 180ms (model via public ingress) | 200-250ms (lookups + model + TLS) |
| Minimum task schedule | N/A (DT scheduler) | 10 seconds (Snowflake hard limit) |

### For This Customer (60 txn/min, $630 avg fraud loss)

**Recommended: Dynamic Tables** unless sub-15s freshness is a hard contractual SLA.

Rationale:
1. **Same warehouse cost** - Both approaches keep the warehouse running 24/7 ($13,190/month)
2. **DT freshness (39s) catches card-testing** - At 60 txn/min, a burst of 5 rapid transactions spans 5 seconds. By the 2nd-3rd transaction (within 39s), the DT has already updated and the model can detect the velocity spike.
3. **Operational risk** - Streams + Tasks introduce failure modes (stream consumption errors, task crashes, 5-min window drift) that require monitoring. DTs are self-healing.
4. **Hybrid Table dependency** - Adds Enterprise Edition requirement and a different storage model to manage.

### If Sub-15s Freshness Becomes a Hard Requirement

Deploy the architecture from this notebook:
1. **Hybrid Tables** for all 5 entities (created in 6.2) - single source of truth
2. **Fast task** (10s, 6.3) - incremental MERGE via stored procedure
3. **Slow task** (5 min, 6.4) - full recompute for window expiry correction
4. **Scoring** reads directly from same Hybrid Tables (NB04, section 4.4)

Total latency budget: 10-15s freshness + 125-150ms scoring = transaction decision within 15 seconds of feature change.

### Cost Summary

| Component | DT Approach (NB02) | Streams + Tasks + Hybrid (this notebook) |
|---|---|---|
| Warehouse (feature refresh) | $13,190/month | $13,190/month |
| SPCS scoring service | $198/month | $198/month |
| Hybrid Table storage | N/A | Included (Enterprise Edition) |
| Engineering maintenance | Near-zero | 0.5 FTE |
| Operational monitoring | Minimal | Required (task + stream alerts) |
| Feature accuracy | Guaranteed (full recompute) | Approximate between slow task runs |
| Feature freshness | 33-39 seconds | 10-15 seconds |
| Scoring lookup latency | 200-500ms (warehouse) | 5-10ms (Hybrid index) |
| **Total platform cost** | **$13,388/month** | **$13,388/month + staff time** |

The cost difference is engineering time, not Snowflake credits. Choose Streams + Tasks only when the 20-25 second improvement in freshness (39s vs 15s) justifies the operational investment.

## 6.2b Backfill Hybrid Tables from Dynamic Tables

Populate the Hybrid Tables with the full feature history computed by the Dynamic Tables in NB02. This gives us realistic data volumes for benchmarking and serves as the initial state before the fast task takes over incremental updates.

| Entity | Rows (from DT) |
|---|---|
| Customer Velocity | 144,017 |
| Merchant Velocity | 5,003 |
| DPAN Velocity | 49,688 |
| IP Velocity | 10,001 |
| Customer-Merchant Velocity | 255,166 |

In [ ]:
-- Backfill Hybrid Tables from Dynamic Tables.
-- Column names differ between DTs and Hybrid Tables in some cases;
-- we map them explicitly here.

-- Entity 1: Customer Velocity (DT columns match Hybrid Table exactly)
TRUNCATE TABLE FRAUD_DEMO_DEV.FEATURES.CUSTOMER_VELOCITY_RT;
INSERT INTO FRAUD_DEMO_DEV.FEATURES.CUSTOMER_VELOCITY_RT (
    customer_id, last_updated_ts, latest_txn_ts,
    purchases_num_l1h, purchases_sum_l1h, purchases_min_l1h, purchases_max_l1h,
    distinct_purchase_amt_l1h, purchases_gbr_num_l1h, purchases_nongbr_num_l1h,
    distinct_card_tokens_l1h, distinct_wallet_dpan_l1h, distinct_merchants_l1h, declined_num_l1h,
    purchases_num_l6h, purchases_sum_l6h, purchases_min_l6h, purchases_max_l6h,
    distinct_purchase_amt_l6h, purchases_gbr_num_l6h, purchases_nongbr_num_l6h,
    distinct_card_tokens_l6h, distinct_wallet_dpan_l6h, distinct_merchants_l6h, declined_num_l6h,
    purchases_num_l24h, purchases_sum_l24h, purchases_min_l24h, purchases_max_l24h,
    distinct_purchase_amt_l24h, purchases_gbr_num_l24h, purchases_nongbr_num_l24h,
    distinct_card_tokens_l24h, distinct_wallet_dpan_l24h, distinct_merchants_l24h, distinct_countries_l24h, declined_num_l24h,
    purchases_num_l48h, purchases_sum_l48h, purchases_min_l48h, purchases_max_l48h,
    distinct_purchase_amt_l48h, purchases_gbr_num_l48h, purchases_nongbr_num_l48h,
    distinct_card_tokens_l48h, distinct_wallet_dpan_l48h, distinct_merchants_l48h, declined_num_l48h,
    purchases_num_l1wk, purchases_sum_l1wk, purchases_min_l1wk, purchases_max_l1wk,
    distinct_purchase_amt_l1wk, purchases_gbr_num_l1wk, purchases_nongbr_num_l1wk,
    distinct_card_tokens_l1wk, distinct_wallet_dpan_l1wk, distinct_merchants_l1wk, distinct_countries_l1wk, declined_num_l1wk
)
SELECT
    customer_id, CURRENT_TIMESTAMP(), latest_txn_ts,
    purchases_num_l1h, purchases_sum_l1h, purchases_min_l1h, purchases_max_l1h,
    distinct_purchase_amt_l1h, purchases_gbr_num_l1h, purchases_nongbr_num_l1h,
    distinct_card_tokens_l1h, distinct_wallet_dpan_l1h, distinct_merchants_l1h, declined_num_l1h,
    purchases_num_l6h, purchases_sum_l6h, purchases_min_l6h, purchases_max_l6h,
    distinct_purchase_amt_l6h, purchases_gbr_num_l6h, purchases_nongbr_num_l6h,
    distinct_card_tokens_l6h, distinct_wallet_dpan_l6h, distinct_merchants_l6h, declined_num_l6h,
    purchases_num_l24h, purchases_sum_l24h, purchases_min_l24h, purchases_max_l24h,
    distinct_purchase_amt_l24h, purchases_gbr_num_l24h, purchases_nongbr_num_l24h,
    distinct_card_tokens_l24h, distinct_wallet_dpan_l24h, distinct_merchants_l24h, distinct_countries_l24h, declined_num_l24h,
    purchases_num_l48h, purchases_sum_l48h, purchases_min_l48h, purchases_max_l48h,
    distinct_purchase_amt_l48h, purchases_gbr_num_l48h, purchases_nongbr_num_l48h,
    distinct_card_tokens_l48h, distinct_wallet_dpan_l48h, distinct_merchants_l48h, declined_num_l48h,
    purchases_num_l1wk, purchases_sum_l1wk, purchases_min_l1wk, purchases_max_l1wk,
    distinct_purchase_amt_l1wk, purchases_gbr_num_l1wk, purchases_nongbr_num_l1wk,
    distinct_card_tokens_l1wk, distinct_wallet_dpan_l1wk, distinct_merchants_l1wk, distinct_countries_l1wk, declined_num_l1wk
FROM FRAUD_DEMO_DEV.FEATURES.CUSTOMER_VELOCITY;

-- Entity 2: Merchant (DT: UNIQUE_CUSTOMERS, MAX_PURCHASE_AMOUNT -> map to DISTINCT_CUSTOMERS, DECLINED=0)
TRUNCATE TABLE FRAUD_DEMO_DEV.FEATURES.MERCHANT_VELOCITY_RT;
INSERT INTO FRAUD_DEMO_DEV.FEATURES.MERCHANT_VELOCITY_RT (
    merchant_id, last_updated_ts, latest_txn_ts,
    merchant_purchases_num_l1h, merchant_purchases_sum_l1h, merchant_distinct_customers_l1h, merchant_declined_num_l1h,
    merchant_purchases_num_l6h, merchant_purchases_sum_l6h, merchant_distinct_customers_l6h, merchant_declined_num_l6h,
    merchant_purchases_num_l24h, merchant_purchases_sum_l24h, merchant_distinct_customers_l24h, merchant_declined_num_l24h,
    merchant_purchases_num_l48h, merchant_purchases_sum_l48h, merchant_distinct_customers_l48h, merchant_declined_num_l48h,
    merchant_purchases_num_l1wk, merchant_purchases_sum_l1wk, merchant_distinct_customers_l1wk, merchant_declined_num_l1wk
)
SELECT
    merchant_id, CURRENT_TIMESTAMP(), NULL,
    merchant_purchases_num_l1h, merchant_purchases_sum_l1h, merchant_unique_customers_l1h, 0,
    merchant_purchases_num_l6h, merchant_purchases_sum_l6h, merchant_unique_customers_l6h, 0,
    merchant_purchases_num_l24h, merchant_purchases_sum_l24h, merchant_unique_customers_l24h, 0,
    merchant_purchases_num_l48h, merchant_purchases_sum_l48h, merchant_unique_customers_l48h, 0,
    merchant_purchases_num_l1wk, merchant_purchases_sum_l1wk, merchant_unique_customers_l1wk, 0
FROM FRAUD_DEMO_DEV.FEATURES.MERCHANT_VELOCITY;

-- Entity 3: DPAN (DT columns match Hybrid Table)
TRUNCATE TABLE FRAUD_DEMO_DEV.FEATURES.WALLET_DPAN_VELOCITY_RT;
INSERT INTO FRAUD_DEMO_DEV.FEATURES.WALLET_DPAN_VELOCITY_RT (
    wallet_dpan, last_updated_ts, latest_txn_ts,
    dpan_purchases_num_l1h, dpan_purchases_sum_l1h, dpan_distinct_merchants_l1h, dpan_declined_num_l1h,
    dpan_purchases_num_l6h, dpan_purchases_sum_l6h, dpan_distinct_merchants_l6h, dpan_declined_num_l6h,
    dpan_purchases_num_l24h, dpan_purchases_sum_l24h, dpan_distinct_merchants_l24h, dpan_declined_num_l24h,
    dpan_purchases_num_l48h, dpan_purchases_sum_l48h, dpan_distinct_merchants_l48h, dpan_declined_num_l48h,
    dpan_purchases_num_l1wk, dpan_purchases_sum_l1wk, dpan_distinct_merchants_l1wk, dpan_declined_num_l1wk
)
SELECT
    wallet_dpan, CURRENT_TIMESTAMP(), NULL,
    dpan_purchases_num_l1h, dpan_purchases_sum_l1h, dpan_distinct_merchants_l1h, dpan_declined_num_l1h,
    dpan_purchases_num_l6h, dpan_purchases_sum_l6h, dpan_distinct_merchants_l6h, dpan_declined_num_l6h,
    dpan_purchases_num_l24h, dpan_purchases_sum_l24h, dpan_distinct_merchants_l24h, dpan_declined_num_l24h,
    dpan_purchases_num_l48h, dpan_purchases_sum_l48h, dpan_distinct_merchants_l48h, dpan_declined_num_l48h,
    dpan_purchases_num_l1wk, dpan_purchases_sum_l1wk, dpan_distinct_merchants_l1wk, dpan_declined_num_l1wk
FROM FRAUD_DEMO_DEV.FEATURES.WALLET_DPAN_VELOCITY;

-- Entity 4: IP (DT: UNIQUE_CUSTOMERS, UNIQUE_CARDS -> map to DISTINCT_CUSTOMERS, DISTINCT_MERCHANTS)
TRUNCATE TABLE FRAUD_DEMO_DEV.FEATURES.IP_VELOCITY_RT;
INSERT INTO FRAUD_DEMO_DEV.FEATURES.IP_VELOCITY_RT (
    ip_address, last_updated_ts, latest_txn_ts,
    ip_purchases_num_l1h, ip_distinct_customers_l1h, ip_distinct_merchants_l1h, ip_declined_num_l1h,
    ip_purchases_num_l6h, ip_distinct_customers_l6h, ip_distinct_merchants_l6h, ip_declined_num_l6h,
    ip_purchases_num_l24h, ip_distinct_customers_l24h, ip_distinct_merchants_l24h, ip_declined_num_l24h,
    ip_purchases_num_l48h, ip_distinct_customers_l48h, ip_distinct_merchants_l48h, ip_declined_num_l48h,
    ip_purchases_num_l1wk, ip_distinct_customers_l1wk, ip_distinct_merchants_l1wk, ip_declined_num_l1wk
)
SELECT
    ip_address, CURRENT_TIMESTAMP(), NULL,
    ip_purchases_num_l1h, ip_unique_customers_l1h, ip_unique_cards_l1h, 0,
    ip_purchases_num_l6h, ip_unique_customers_l6h, ip_unique_cards_l6h, 0,
    ip_purchases_num_l24h, ip_unique_customers_l24h, ip_unique_cards_l24h, 0,
    ip_purchases_num_l48h, ip_unique_customers_l48h, ip_unique_cards_l48h, 0,
    ip_purchases_num_l1wk, ip_unique_customers_l1wk, ip_unique_cards_l1wk, 0
FROM FRAUD_DEMO_DEV.FEATURES.IP_VELOCITY;

-- Entity 5: Customer-Merchant (DT has DECLINED columns, Hybrid Table doesn't - skip them)
TRUNCATE TABLE FRAUD_DEMO_DEV.FEATURES.CUSTOMER_MERCHANT_VELOCITY_RT;
INSERT INTO FRAUD_DEMO_DEV.FEATURES.CUSTOMER_MERCHANT_VELOCITY_RT (
    customer_id, merchant_id, last_updated_ts, latest_txn_ts,
    cust_merch_purchases_num_l1h, cust_merch_purchases_sum_l1h,
    cust_merch_purchases_num_l6h, cust_merch_purchases_sum_l6h,
    cust_merch_purchases_num_l24h, cust_merch_purchases_sum_l24h,
    cust_merch_purchases_num_l48h, cust_merch_purchases_sum_l48h,
    cust_merch_purchases_num_l1wk, cust_merch_purchases_sum_l1wk
)
SELECT
    customer_id, merchant_id, CURRENT_TIMESTAMP(), NULL,
    cust_merch_purchases_num_l1h, cust_merch_purchases_sum_l1h,
    cust_merch_purchases_num_l6h, cust_merch_purchases_sum_l6h,
    cust_merch_purchases_num_l24h, cust_merch_purchases_sum_l24h,
    cust_merch_purchases_num_l48h, cust_merch_purchases_sum_l48h,
    cust_merch_purchases_num_l1wk, cust_merch_purchases_sum_l1wk
FROM FRAUD_DEMO_DEV.FEATURES.CUSTOMER_MERCHANT_VELOCITY;

-- Verify
SELECT 'CUSTOMER_VELOCITY_RT' AS entity, COUNT(*) AS row_count FROM FRAUD_DEMO_DEV.FEATURES.CUSTOMER_VELOCITY_RT
UNION ALL SELECT 'MERCHANT_VELOCITY_RT', COUNT(*) FROM FRAUD_DEMO_DEV.FEATURES.MERCHANT_VELOCITY_RT
UNION ALL SELECT 'WALLET_DPAN_VELOCITY_RT', COUNT(*) FROM FRAUD_DEMO_DEV.FEATURES.WALLET_DPAN_VELOCITY_RT
UNION ALL SELECT 'IP_VELOCITY_RT', COUNT(*) FROM FRAUD_DEMO_DEV.FEATURES.IP_VELOCITY_RT
UNION ALL SELECT 'CUSTOMER_MERCHANT_VELOCITY_RT', COUNT(*) FROM FRAUD_DEMO_DEV.FEATURES.CUSTOMER_MERCHANT_VELOCITY_RT;